In [13]:
# ============================================================
# AUTOMATICALLY FIND FINAL DATASET
# ============================================================
import os
import glob
import numpy as np

print("=" * 80)
print("SEARCHING KAGGLE INPUT FOR DATASET")
print("=" * 80)

train_dirs = []

for root, dirs, files in os.walk("/kaggle/input"):

    for d in dirs:

        if d.lower() == "train":

            train_dirs.append(
                os.path.join(root, d)
            )


print("\nTrain directories found:")

for path in train_dirs:

    print("  ", path)

if len(train_dirs) == 0:

    raise FileNotFoundError(
        "Could not find a 'train' directory inside /kaggle/input"
    )

elif len(train_dirs) > 1:

    print(
        "\nWARNING: More than one train directory found."
    )

    for i, path in enumerate(train_dirs):

        print(i, path)

    raise RuntimeError(
        "Multiple train directories found. "
        "We need to identify the correct dataset."
    )


# ------------------------------------------------------------
# TRAIN directory
# ------------------------------------------------------------

TRAIN_ROOT = train_dirs[0]

# ------------------------------------------------------------
# DATASET ROOT
# ------------------------------------------------------------

DATA_ROOT = os.path.dirname(
    TRAIN_ROOT
)


print("\n" + "=" * 80)

print(
    "DATASET ROOT:"
)

print(
    DATA_ROOT
)

print(
    "\nTRAIN:"
)

print(
    TRAIN_ROOT
)

print(
    "\nVALIDATION:"
)

print(
    os.path.join(
        DATA_ROOT,
        "validation"
    )
)

print(
    "\nTEST:"
)

print(
    os.path.join(
        DATA_ROOT,
        "test"
    )
)

print("=" * 80)

SEARCHING KAGGLE INPUT FOR DATASET

Train directories found:
   /kaggle/input/datasets/beginne/cnn1final/FINALCNNDATA_NEW_01/train

DATASET ROOT:
/kaggle/input/datasets/beginne/cnn1final/FINALCNNDATA_NEW_01

TRAIN:
/kaggle/input/datasets/beginne/cnn1final/FINALCNNDATA_NEW_01/train

VALIDATION:
/kaggle/input/datasets/beginne/cnn1final/FINALCNNDATA_NEW_01/validation

TEST:
/kaggle/input/datasets/beginne/cnn1final/FINALCNNDATA_NEW_01/test


In [14]:
# ============================================================
# LOAD FINAL DATASET + CHECK CLASS DISTRIBUTION
# ============================================================
import os
import glob
import numpy as np
DATA_ROOT = "/kaggle/input/datasets/beginne/cnn1final/FINALCNNDATA_NEW_01"
def collect_split(split_name):
    split_root = os.path.join(
        DATA_ROOT,
        split_name
    )
    site_dir = os.path.join(
        split_root,
        "site"
    )
    no_site_dir = os.path.join(
        split_root,
        "no_site"
    )
    site_files = sorted(
        glob.glob(os.path.join(site_dir, "*.tif")) +
        glob.glob(os.path.join(site_dir, "*.tiff"))
    )
    no_site_files = sorted(
        glob.glob(os.path.join(no_site_dir, "*.tif")) +
        glob.glob(os.path.join(no_site_dir, "*.tiff"))
    )
    filepaths = site_files + no_site_files

    labels = np.array(
        [1] * len(site_files) +
        [0] * len(no_site_files),
        dtype=np.int64
    )
    return filepaths, labels
# ============================================================
# LOAD EACH SPLIT
# ============================================================
train_fp, train_labels = collect_split("train")
val_fp, val_labels = collect_split("validation")
test_fp, test_labels = collect_split("test")
# ============================================================
# COMBINE TRAIN + VALIDATION
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# COMBINE TRAIN + VALIDATION PATHS
# ------------------------------------------------------------

trainval_fp = train_fp + val_fp

trainval_labels = np.concatenate([
    train_labels,
    val_labels
])


# ------------------------------------------------------------
# LOAD METADATA
# ------------------------------------------------------------

METADATA_PATH = (
    "/kaggle/input/datasets/beginne/"
    "cnn1final/FINALCNNDATA_NEW_01/"
    "metadata/combined/metadata_all.csv"
)

metadata_df = pd.read_csv(
    METADATA_PATH
)

print("=" * 80)
print("ALIGNING MONUMENT COUNTS")
print("=" * 80)

print(
    "Metadata rows:",
    len(metadata_df)
)


# ------------------------------------------------------------
# CREATE PATCH-NAME → MONUMENT-COUNT LOOKUP
# ------------------------------------------------------------

metadata_df["patch_name"] = (
    metadata_df["patch_name"]
    .astype(str)
    .apply(os.path.basename)
)

metadata_df["monument_count"] = (
    pd.to_numeric(
        metadata_df["monument_count"],
        errors="coerce"
    )
    .fillna(0)
    .astype(np.int64)
)

monument_count_lookup = dict(
    zip(
        metadata_df["patch_name"],
        metadata_df["monument_count"]
    )
)


# ------------------------------------------------------------
# MATCH MONUMENT COUNTS TO TRAIN+VAL PATH ORDER
# ------------------------------------------------------------

trainval_monument_counts = np.array(
    [
        monument_count_lookup[
            os.path.basename(fp)
        ]
        for fp in trainval_fp
    ],
    dtype=np.int64
)

# ------------------------------------------------------------
# CONVERT LABELS TO NUMPY ARRAY
# ------------------------------------------------------------

trainval_lb_arr = np.asarray(
    trainval_labels,
    dtype=np.int64
)

# ------------------------------------------------------------
# CHECK ALIGNMENT
# ------------------------------------------------------------

assert len(trainval_fp) == len(trainval_lb_arr)

assert len(trainval_fp) == len(
    trainval_monument_counts
)


# ------------------------------------------------------------
# PRINT CHECKS
# ------------------------------------------------------------

print(
    f"Train+Val samples       : {len(trainval_fp)}"
)

print(
    f"Train+Val sites         : "
    f"{np.sum(trainval_lb_arr == 1)}"
)

print(
    f"Train+Val no-site       : "
    f"{np.sum(trainval_lb_arr == 0)}"
)

print(
    f"Monument counts aligned : "
    f"{len(trainval_monument_counts)}"
)

print(
    f"Maximum monument count  : "
    f"{np.max(trainval_monument_counts)}"
)

print(
    f"Total monuments         : "
    f"{np.sum(trainval_monument_counts)}"
)

print()
print("Label alignment          : PASS")
print("Monument-count alignment : PASS")


# ============================================================
# DISTRIBUTION FUNCTION
# ============================================================

def show_distribution(name, labels):

    total = len(labels)

    positives = int(
        np.sum(labels == 1)
    )

    negatives = int(
        np.sum(labels == 0)
    )

    print(
        f"{name:15s} | "
        f"Total = {total:5d} | "
        f"Site = {positives:5d} | "
        f"No-site = {negatives:5d} | "
        f"Site % = {100 * positives / total:.2f}%"
    )


# ============================================================
# FINAL DATASET DISTRIBUTION
# ============================================================

print()
print("=" * 80)
print("FINAL DATASET DISTRIBUTION")
print("=" * 80)

show_distribution(
    "TRAIN",
    train_labels
)

show_distribution(
    "VALIDATION",
    val_labels
)

show_distribution(
    "TRAIN + VAL",
    trainval_labels
)

show_distribution(
    "TEST",
    test_labels
)

print("=" * 80)

ALIGNING MONUMENT COUNTS
Metadata rows: 3617
Train+Val samples       : 3117
Train+Val sites         : 946
Train+Val no-site       : 2171
Monument counts aligned : 3117
Maximum monument count  : 42
Total monuments         : 2541

Label alignment          : PASS
Monument-count alignment : PASS

FINAL DATASET DISTRIBUTION
TRAIN           | Total =  2533 | Site =   769 | No-site =  1764 | Site % = 30.36%
VALIDATION      | Total =   584 | Site =   177 | No-site =   407 | Site % = 30.31%
TRAIN + VAL     | Total =  3117 | Site =   946 | No-site =  2171 | Site % = 30.35%
TEST            | Total =   500 | Site =   150 | No-site =   350 | Site % = 30.00%


In [15]:
# ============================================================
# IMPORTS + GPU SETUP
# ============================================================
import os
import glob
import random
import copy

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from sklearn.model_selection import (
    StratifiedGroupKFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 80)
print("DEVICE")
print("=" * 80)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Device :",
    DEVICE
)


if DEVICE.type == "cuda":

    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM   :",
        round(
            torch.cuda.get_device_properties(0)
            .total_memory / 1024**3,
            2
        ),
        "GB"
    )

else:

    raise RuntimeError(
        "CUDA GPU not detected."
    )


print("=" * 80)

DEVICE
PyTorch: 2.10.0+cu128
Device : cuda
GPU    : Tesla T4
VRAM   : 14.56 GB


In [16]:
# ============================================================
# FINAL V5 EXPERIMENT CONFIGURATION
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # DATA
    # --------------------------------------------------------

    "n_channels": 11,
    "n_handcrafted": 15,
    "patch_size": 250,

    # --------------------------------------------------------
    # MODEL — V5
    # --------------------------------------------------------

    "dropout": 0.35,

    "use_se_blocks": True,

    "use_handcrafted_features": True,

    # --------------------------------------------------------
    # CHANNEL DROPOUT
    # --------------------------------------------------------

    "use_channel_dropout": True,

    "channel_dropout_prob": 0.5,

    "channel_dropout_max_channels": 2,


    # --------------------------------------------------------
    # CLASS IMBALANCE
    # --------------------------------------------------------
    "use_balanced_sampler": True,
    "use_pos_weight": False,


    # --------------------------------------------------------
    # SOFT LABELS
    # --------------------------------------------------------

    "negative_soft_label": 0.05,
    "positive_soft_label_cap": 0.95,


    # --------------------------------------------------------
    # SPATIAL 5-FOLD CV
    # --------------------------------------------------------

    "n_folds": 5,
    "fold_seed": 658,


    # --------------------------------------------------------
    # SUPERVISED TRAINING
    # --------------------------------------------------------

    "batch_size": 16,

    "epochs": 30,

    "learning_rate": 3e-4,

    "weight_decay": 1e-4,

    "grad_clip_norm": 1.0,


    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    "scheduler_factor": 0.5,
    "scheduler_patience": 3,


    # --------------------------------------------------------
    # CHECKPOINT / EARLY STOPPING
    # --------------------------------------------------------
    "checkpoint_metric": "auc",
    "early_stopping_patience": 8,


    # --------------------------------------------------------
    # TTA
    # --------------------------------------------------------

    "use_tta": True,


    # --------------------------------------------------------
    # THRESHOLD SEARCH
    # --------------------------------------------------------

    "threshold_grid": np.arange(
        0.10,
        0.71,
        0.01
    ),


    # --------------------------------------------------------
    # OPTIONAL SSL
    # --------------------------------------------------------
    "use_ssl_pretrain": False,

    "ssl_checkpoint_path":
        "/kaggle/working/ssl_encoder.pt",


    # --------------------------------------------------------
    # CHECKPOINTS
    # --------------------------------------------------------

    "checkpoint_dir":
        "/kaggle/working/cnn_v5_checkpoints",

}


os.makedirs(
    CONFIG["checkpoint_dir"],
    exist_ok=True
)


print("=" * 80)
print("FINAL V5 CONFIGURATION")
print("=" * 80)

for key, value in CONFIG.items():

    print(
        f"{key:35s}: {value}"
    )

print("=" * 80)

FINAL V5 CONFIGURATION
n_channels                         : 11
n_handcrafted                      : 15
patch_size                         : 250
dropout                            : 0.35
use_se_blocks                      : True
use_handcrafted_features           : True
use_channel_dropout                : True
channel_dropout_prob               : 0.5
channel_dropout_max_channels       : 2
use_balanced_sampler               : True
use_pos_weight                     : False
negative_soft_label                : 0.05
positive_soft_label_cap            : 0.95
n_folds                            : 5
fold_seed                          : 658
batch_size                         : 16
epochs                             : 30
learning_rate                      : 0.0003
weight_decay                       : 0.0001
grad_clip_norm                     : 1.0
scheduler_factor                   : 0.5
scheduler_patience                 : 3
checkpoint_metric                  : auc
early_stopping_patience      

In [17]:
# ============================================================
# FIND METADATA CSV FILES
# ============================================================

import pandas as pd
import os
import glob


csv_files = glob.glob(
    os.path.join(
        DATA_ROOT,
        "**",
        "*.csv"
    ),
    recursive=True
)

print("=" * 80)
print("CSV FILES FOUND")
print("=" * 80)

for f in csv_files:

    print(
        os.path.relpath(
            f,
            DATA_ROOT
        )
    )

print("=" * 80)

CSV FILES FOUND
metadata/dataset_summary.csv
metadata/combined/metadata_validation.csv
metadata/combined/metadata_all.csv
metadata/combined/metadata_train.csv
metadata/combined/metadata_test.csv


In [18]:
# ============================================================
# LOAD METADATA
# ============================================================

def find_csv(filename):

    matches = [
        f for f in csv_files
        if os.path.basename(f).lower()
        == filename.lower()
    ]

    if len(matches) == 0:

        raise FileNotFoundError(
            f"Could not find {filename}"
        )

    if len(matches) > 1:

        print(
            f"Multiple copies of {filename} found:"
        )

        for m in matches:
            print("   ", m)

        raise RuntimeError(
            f"Multiple {filename} files found."
        )

    return matches[0]


metadata_all_path = find_csv(
    "metadata_all.csv"
)


metadata_all = pd.read_csv(
    metadata_all_path
)


print("=" * 80)
print("METADATA")
print("=" * 80)

print(
    "Path:",
    metadata_all_path
)

print(
    "Shape:",
    metadata_all.shape
)

print(
    "\nColumns:"
)

for col in metadata_all.columns:

    print(
        "  ",
        col
    )

print("=" * 80)

METADATA
Path: /kaggle/input/datasets/beginne/cnn1final/FINALCNNDATA_NEW_01/metadata/combined/metadata_all.csv
Shape: (3617, 18)

Columns:
   patch_name
   source_path
   final_path
   split
   label
   label_name
   monument_ids
   monument_count
   group_id
   width
   height
   bands
   epsg
   bounds_left
   bounds_bottom
   bounds_right
   bounds_top
   geometry_wkt


In [20]:
# ============================================================
# IDENTIFY METADATA COLUMNS
# ============================================================

columns_lower = {
    col.lower(): col
    for col in metadata_all.columns
}


def find_column(
    candidates,
    required=True
):

    for candidate in candidates:

        if candidate.lower() in columns_lower:

            return columns_lower[
                candidate.lower()
            ]

    for col in metadata_all.columns:
        col_lower = col.lower()

        for candidate in candidates:
            if candidate.lower() in col_lower:

                return col


    if required:

        raise KeyError(
            "Could not identify column. "
            f"Tried: {candidates}\n"
            f"Available: {list(metadata_all.columns)}"
        )

    return None


PATCH_COL = find_column([
    "patch_name",
    "patch",
    "filename",
    "file_name",
    "tif_name",
    "name"
])


LABEL_COL = find_column([
    "label"
])


MONUMENT_COUNT_COL = find_column([
    "monument_count",
    "monument_counts",
    "n_monuments",
    "num_monuments",
    "number_of_monuments"
])


GROUP_COL = find_column([
    "spatial_group_id",
    "spatial_group",
    "group_id",
    "group"
])


print("=" * 80)
print("IDENTIFIED METADATA COLUMNS")
print("=" * 80)

print(
    "Patch column          :",
    PATCH_COL
)

print(
    "Label column          :",
    LABEL_COL
)

print(
    "Monument count column :",
    MONUMENT_COUNT_COL
)

print(
    "Spatial group column  :",
    GROUP_COL
)

print("=" * 80)

IDENTIFIED METADATA COLUMNS
Patch column          : patch_name
Label column          : label
Monument count column : monument_count
Spatial group column  : group_id


In [21]:
# ============================================================
# CLEAN METADATA + BUILD LOOKUP
# ============================================================

metadata_all = metadata_all.copy()

metadata_all[MONUMENT_COUNT_COL] = (
    pd.to_numeric(
        metadata_all[MONUMENT_COUNT_COL],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)


metadata_all[LABEL_COL] = (
    pd.to_numeric(
        metadata_all[LABEL_COL],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

metadata_all["_basename"] = (
    metadata_all[PATCH_COL]
    .astype(str)
    .apply(os.path.basename)
)


metadata_lookup = {}

for _, row in metadata_all.iterrows():

    key = row["_basename"]

    metadata_lookup[key] = {
        "label": int(
            row[LABEL_COL]
        ),

        "monument_count": int(
            row[MONUMENT_COUNT_COL]
        ),

        "group": row[GROUP_COL]
    }


print("=" * 80)
print("METADATA LOOKUP")
print("=" * 80)

print(
    "Metadata records:",
    len(metadata_lookup)
)

print(
    "Unique groups:",
    metadata_all[GROUP_COL].nunique()
)

print(
    "Positive records:",
    (metadata_all[LABEL_COL] == 1).sum()
)

print(
    "Negative records:",
    (metadata_all[LABEL_COL] == 0).sum()
)

print("=" * 80)

METADATA LOOKUP
Metadata records: 3617
Unique groups: 40
Positive records: 1096
Negative records: 2521


In [22]:
# ============================================================
# BUILD CV METADATA ARRAYS
# ============================================================

cv_monument_counts = []
cv_groups = []
cv_labels_check = []

missing_metadata = []


for filepath, label in zip(
    trainval_fp,
    trainval_labels
):

    basename = os.path.basename(
        filepath
    )


    if basename not in metadata_lookup:

        missing_metadata.append(
            basename
        )

        continue


    info = metadata_lookup[
        basename
    ]


    cv_monument_counts.append(
        info["monument_count"]
    )

    cv_groups.append(
        info["group"]
    )

    cv_labels_check.append(
        info["label"]
    )

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

if missing_metadata:

    print(
        "Missing metadata:",
        len(missing_metadata)
    )

    print(
        missing_metadata[:20]
    )

    raise RuntimeError(
        "Some CV patches could not be matched "
        "to metadata."
    )


cv_monument_counts = np.asarray(
    cv_monument_counts,
    dtype=np.int64
)

cv_groups = np.asarray(
    cv_groups
)

cv_labels_check = np.asarray(
    cv_labels_check,
    dtype=np.int64
)


assert len(cv_monument_counts) == len(
    trainval_fp
)

assert len(cv_groups) == len(
    trainval_fp
)

label_mismatches = np.sum(
    cv_labels_check != trainval_labels
)


print("=" * 80)
print("CV METADATA MATCH")
print("=" * 80)

print(
    "CV samples:",
    len(trainval_fp)
)

print(
    "Missing metadata:",
    len(missing_metadata)
)

print(
    "Label mismatches:",
    label_mismatches
)

print(
    "Unique spatial groups:",
    len(np.unique(cv_groups))
)

print(
    "Maximum monument count:",
    cv_monument_counts.max()
)

print("=" * 80)


if label_mismatches != 0:

    raise RuntimeError(
        "Folder labels and metadata labels disagree."
    )

CV METADATA MATCH
CV samples: 3117
Missing metadata: 0
Label mismatches: 0
Unique spatial groups: 34
Maximum monument count: 42


In [23]:
# ============================================================
# MONUMENT-COUNT SOFT LABELS
# ============================================================

SOFT_LABEL_MAP = {

    0: 0.05,

    1: 0.7377,

    2: 0.8212,

    3: 0.8719,

    4: 0.9026,

    5: 0.9213,

    6: 0.9315,

    7: 0.9380,

    8: 0.9425,

    9: 0.9455,

    10: 0.9476,
}


def monument_count_to_soft_label(
    count
):

    count = int(count)


    if count <= 0:

        return 0.05


    if count >= 20:

        return 0.95


    if count in SOFT_LABEL_MAP:

        return SOFT_LABEL_MAP[count]

    return float(
        np.interp(
            count,
            [10, 20],
            [0.9476, 0.95]
        )
    )


cv_soft_labels = np.array([
    monument_count_to_soft_label(
        count
    )
    for count in cv_monument_counts
], dtype=np.float32)


print("=" * 80)
print("SOFT LABEL CHECK")
print("=" * 80)

for count in [
    0, 1, 2, 3, 4, 5,
    10, 15, 20, 30
]:

    print(
        f"Monuments = {count:2d}"
        f"  →  target = "
        f"{monument_count_to_soft_label(count):.4f}"
    )


print("\nDistribution:")

print(
    "Mean  :",
    cv_soft_labels.mean()
)

print(
    "Median:",
    np.median(cv_soft_labels)
)

print(
    "Min   :",
    cv_soft_labels.min()
)

print(
    "Max   :",
    cv_soft_labels.max()
)

print("=" * 80)

SOFT LABEL CHECK
Monuments =  0  →  target = 0.0500
Monuments =  1  →  target = 0.7377
Monuments =  2  →  target = 0.8212
Monuments =  3  →  target = 0.8719
Monuments =  4  →  target = 0.9026
Monuments =  5  →  target = 0.9213
Monuments = 10  →  target = 0.9476
Monuments = 15  →  target = 0.9488
Monuments = 20  →  target = 0.9500
Monuments = 30  →  target = 0.9500

Distribution:
Mean  : 0.27733082
Median: 0.05
Min   : 0.05
Max   : 0.95


In [25]:
# ============================================================
# V5 DATASET
# ============================================================

class TerrainPatchDataset(Dataset):

    def __init__(
        self,
        filepaths,
        hard_labels,
        monument_counts,
        mean,
        std,
        hc_mean,
        hc_std,
        augment=False,
        use_handcrafted=True,
        use_channel_dropout=True,
        channel_dropout_prob=0.5,
        channel_dropout_max_channels=2
    ):

        self.filepaths = filepaths

        self.hard_labels = np.asarray(
            hard_labels,
            dtype=np.int64
        )

        self.monument_counts = np.asarray(
            monument_counts,
            dtype=np.int64
        )

        self.mean = mean.reshape(
            -1, 1, 1
        )

        self.std = std.reshape(
            -1, 1, 1
        )

        self.hc_mean = hc_mean

        self.hc_std = hc_std

        self.augment = augment

        self.use_handcrafted = (
            use_handcrafted
        )

        self.use_channel_dropout = (
            use_channel_dropout
        )

        self.channel_dropout_prob = (
            channel_dropout_prob
        )

        self.channel_dropout_max_channels = (
            channel_dropout_max_channels
        )


    def __len__(self):

        return len(
            self.filepaths
        )


    @staticmethod
    def _geometric_augment(
        arr
    ):

        if random.random() < 0.5:

            arr = arr[
                :,
                :,
                ::-1
            ]


        if random.random() < 0.5:

            arr = arr[
                :,
                ::-1,
                :
            ]


        k = random.choice(
            [0, 1, 2, 3]
        )


        if k != 0:

            arr = np.rot90(
                arr,
                k=k,
                axes=(1, 2)
            )


        return arr

    def _channel_dropout(
        self,
        arr
    ):

        if random.random() < (
            self.channel_dropout_prob
        ):

            n_drop = random.randint(
                1,
                self.channel_dropout_max_channels
            )


            drop_idx = random.sample(
                range(arr.shape[0]),
                n_drop
            )


            # 0 in normalized space =
            # dataset mean in raw space.

            arr[
                drop_idx
            ] = 0.0


        return arr


    def __getitem__(
        self,
        idx
    ):

        filepath = self.filepaths[
            idx
        ]


        hard_label = int(
            self.hard_labels[idx]
        )


        monument_count = int(
            self.monument_counts[idx]
        )


        # ----------------------------------------------------
        # RAW PATCH
        # ----------------------------------------------------
        raw_arr = load_patch(
            filepath
        )
        # ----------------------------------------------------
        # HANDCRAFTED FEATURES
        # ----------------------------------------------------
        if self.use_handcrafted:
            hc_feat = (
                compute_handcrafted_features(
                    raw_arr
                )
            )

            hc_feat = (
                hc_feat
                - self.hc_mean
            ) / self.hc_std

        else:

            hc_feat = np.zeros(
                1,
                dtype=np.float32
            )

        # ----------------------------------------------------
        # GEOMETRIC AUGMENTATION
        # ----------------------------------------------------

        arr = raw_arr

        if self.augment:

            arr = (
                self._geometric_augment(
                    arr
                )
            )


        arr = np.ascontiguousarray(
            arr
        )


        # ----------------------------------------------------
        # NORMALIZE
        # ----------------------------------------------------

        arr = (
            arr - self.mean
        ) / self.std


        # ----------------------------------------------------
        # CHANNEL DROPOUT
        # ----------------------------------------------------

        if (
            self.augment
            and self.use_channel_dropout
        ):

            arr = (
                self._channel_dropout(
                    arr
                )
            )


        # ----------------------------------------------------
        # SOFT LABEL
        # ----------------------------------------------------

        soft_label = (
            monument_count_to_soft_label(
                monument_count
            )
        )


        return (

            torch.from_numpy(
                arr
            ).float(),

            torch.from_numpy(
                hc_feat
            ).float(),

            torch.tensor(
                soft_label,
                dtype=torch.float32
            ),

            torch.tensor(
                hard_label,
                dtype=torch.long
            )
        )

In [26]:
# ============================================================
# TIFF / RASTER LOADER
# ============================================================

def load_patch(filepath):

    if filepath.lower().endswith(".npy"):

        arr = np.load(
            filepath
        ).astype(
            np.float32
        )

    else:

        with rasterio.open(
            filepath
        ) as src:

            arr = src.read().astype(
                np.float32
            )


    # --------------------------------------------------------
    # Clean invalid values
    # --------------------------------------------------------

    arr = np.nan_to_num(
        arr,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


    # --------------------------------------------------------
    # Shape check
    # --------------------------------------------------------

    expected_shape = (
        CONFIG["n_channels"],
        CONFIG["patch_size"],
        CONFIG["patch_size"]
    )


    if arr.shape != expected_shape:

        raise ValueError(
            f"\nUnexpected patch shape.\n"
            f"File: {filepath}\n"
            f"Got: {arr.shape}\n"
            f"Expected: {expected_shape}"
        )


    return arr

In [27]:
# ============================================================
# CHANNEL NORMALIZATION
# ============================================================

def compute_channel_stats(
    filepaths,
    max_samples=800
):

    if len(filepaths) <= max_samples:

        sampled_files = filepaths

    else:

        sampled_files = random.sample(
            filepaths,
            max_samples
        )


    n_channels = CONFIG[
        "n_channels"
    ]


    channel_sum = np.zeros(
        n_channels,
        dtype=np.float64
    )

    channel_sum_sq = np.zeros(
        n_channels,
        dtype=np.float64
    )

    total_pixels = 0


    for filepath in tqdm(
        sampled_files,
        desc="Computing channel statistics"
    ):

        arr = load_patch(
            filepath
        )

        pixels = arr.reshape(
            n_channels,
            -1
        )

        channel_sum += (
            pixels.sum(axis=1)
        )

        channel_sum_sq += (
            (pixels ** 2).sum(axis=1)
        )

        total_pixels += (
            pixels.shape[1]
        )

    mean = (
        channel_sum
        / total_pixels
    )


    variance = (
        channel_sum_sq
        / total_pixels
        - mean ** 2
    )


    variance = np.maximum(
        variance,
        1e-8
    )


    std = np.sqrt(
        variance
    )

    std[std < 1e-6] = 1.0

    return (
        mean.astype(np.float32),
        std.astype(np.float32)
    )

CHANNEL_MEAN, CHANNEL_STD = (
    compute_channel_stats(
        trainval_fp
    )
)
print("\n" + "=" * 80)
print("CHANNEL NORMALIZATION")
print("=" * 80)


for i in range(
    CONFIG["n_channels"]
):
    print(
        f"Channel {i+1:02d}: "
        f"mean={CHANNEL_MEAN[i]:.6f} | "
        f"std={CHANNEL_STD[i]:.6f}"
    )

print("=" * 80)

Computing channel statistics:   0%|          | 0/800 [00:00<?, ?it/s]


CHANNEL NORMALIZATION
Channel 01: mean=-11.641353 | std=341.507141
Channel 02: mean=-11.631943 | std=341.507446
Channel 03: mean=-11.640773 | std=341.507172
Channel 04: mean=-11.415938 | std=341.514862
Channel 05: mean=-11.665041 | std=341.506775
Channel 06: mean=-4.334789 | std=341.816254
Channel 07: mean=-10.747318 | std=341.537689
Channel 08: mean=-21.526899 | std=470.228333
Channel 09: mean=-21.530003 | std=470.228180
Channel 10: mean=-21.528440 | std=470.228241
Channel 11: mean=-21.525211 | std=470.228394


In [28]:
def compute_handcrafted_features(
    arr
):

    LRM = arr[0]
    SVF = arr[1]
    SLOPE = arr[2]
    LOCAL_DOM = arr[10]
    R = arr[6]
    NIR = arr[9]
    feats = []

    feats += [

        LRM.mean(),

        LRM.std(),

        LRM.min(),

        LRM.max(),

        LRM.max() - LRM.min()

    ]


    gy, gx = np.gradient(
        LRM
    )


    edge_mag = np.sqrt(
        gx ** 2
        +
        gy ** 2
    )


    feats += [

        edge_mag.mean(),

        edge_mag.std()

    ]

    feats += [

        SLOPE.mean(),

        SLOPE.max()

    ]
    feats += [

        SVF.mean(),

        SVF.std()

    ]

    feats += [

        LOCAL_DOM.mean(),

        LOCAL_DOM.std()

    ]
    ndvi = (
        (NIR - R)
        /
        (NIR + R + 1e-6)
    )


    feats += [

        ndvi.mean(),

        ndvi.std()

    ]

    return np.asarray(
        feats,
        dtype=np.float32
    )

In [29]:
def compute_handcrafted_stats(
    filepaths,
    max_samples=800
):

    if len(filepaths) <= max_samples:
        sampled_files = filepaths

    else:
        sampled_files = random.sample(
            filepaths,
            max_samples
        )


    all_features = np.zeros(
        (
            len(sampled_files),
            CONFIG["n_handcrafted"]
        ),
        dtype=np.float64
    )


    for i, filepath in enumerate(
        tqdm(
            sampled_files,
            desc="Computing handcrafted statistics"
        )
    ):

        arr = load_patch(
            filepath
        )


        all_features[i] = (
            compute_handcrafted_features(
                arr
            )
        )


    mean = all_features.mean(
        axis=0
    )

    std = all_features.std(
        axis=0
    )


    std[std < 1e-6] = 1.0


    return (
        mean.astype(np.float32),
        std.astype(np.float32)
    )


HANDCRAFTED_MEAN, HANDCRAFTED_STD = (
    compute_handcrafted_stats(
        trainval_fp
    )
)

print("=" * 80)
print("HANDCRAFTED FEATURES")
print("=" * 80)

for i in range(
    CONFIG["n_handcrafted"]
):

    print(
        f"Feature {i+1:02d}: "
        f"mean={HANDCRAFTED_MEAN[i]:.6f} | "
        f"std={HANDCRAFTED_STD[i]:.6f}"
    )


print("=" * 80)

Computing handcrafted statistics:   0%|          | 0/800 [00:00<?, ?it/s]

HANDCRAFTED FEATURES
Feature 01: mean=-12.247602 | std=83.149796
Feature 02: mean=59.303333 | std=335.026367
Feature 03: mean=-399.942230 | std=1959.399414
Feature 04: mean=0.068850 | std=0.028494
Feature 05: mean=400.011078 | std=1959.399414
Feature 06: mean=0.772917 | std=4.447974
Feature 07: mean=12.926925 | std=66.431335
Feature 08: mean=-12.246861 | std=83.149673
Feature 09: mean=0.082200 | std=0.027501
Feature 10: mean=-12.238085 | std=83.149841
Feature 11: mean=59.309010 | std=335.025665
Feature 12: mean=-21.374294 | std=87.934723
Feature 13: mean=163.799637 | std=430.186798
Feature 14: mean=-0.191167 | std=0.032139
Feature 15: mean=0.071257 | std=0.029984


In [30]:
sample_arr = load_patch(
    trainval_fp[0]
)
sample_features = (
    compute_handcrafted_features(
        sample_arr
    )
)
print(
    "Raster shape:",
    sample_arr.shape
)
print(
    "Handcrafted feature shape:",
    sample_features.shape
)

print(
    "Expected:",
    (CONFIG["n_handcrafted"],)
)
assert sample_features.shape == (
    CONFIG["n_handcrafted"],
)
assert np.all(
    np.isfinite(
        sample_features
    )
)
print(
    "\nHandcrafted feature sanity check PASSED."
)

Raster shape: (11, 250, 250)
Handcrafted feature shape: (15,)
Expected: (15,)

Handcrafted feature sanity check PASSED.


In [40]:
# ============================================================
# CNN_BASE2 (256 CHANNELS) + MOBILEVIT-v2 SANITY CHECK
# ============================================================

class CNNBase2Encoder(nn.Module):

    def __init__(
        self,
        in_channels=11,
        use_se_blocks=True,
    ):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=5,
                padding=2,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(
                128,
                256,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x


# ============================================================
# HYBRID MODEL
# ============================================================

class CNNBase2MobileViTv2(nn.Module):

    def __init__(
        self,
        in_channels=11,
        n_handcrafted=15,
        dropout=0.35,
        use_se_blocks=True,
        use_handcrafted=True,
    ):
        super().__init__()

        # EXACT 256-CHANNEL CNN BASE
        self.encoder = CNNBase2Encoder(
            in_channels=in_channels,
            use_se_blocks=use_se_blocks,
        )

        self.dropout = nn.Dropout2d(
            dropout
        )

        # MobileViT receives the FINAL CNN feature map
        self.mobilevit = MobileViTv2Block(
            in_channels=256,
            transformer_dim=128,
            patch_size=2,
            num_heads=4,
            num_transformer_blocks=2,
            mlp_ratio=2.0,
            dropout=0.10,
        )

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.use_handcrafted = (
            use_handcrafted
        )

        fusion_dim = (
            256
            +
            (
                n_handcrafted
                if use_handcrafted
                else 0
            )
        )

        self.head = nn.Sequential(

            nn.Linear(
                fusion_dim,
                64,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                64,
                1,
            ),
        )

    def forward(
        self,
        x,
        feat=None,
    ):

        # ----------------------------------------------------
        # CNN BASE2
        # ----------------------------------------------------

        x = self.encoder(x)

        # ----------------------------------------------------
        # FINAL CNN FEATURE MAP
        # [B, 256, 62, 62]
        # ----------------------------------------------------

        x = self.dropout(x)

        # ----------------------------------------------------
        # MOBILEVIT
        # [B, 256, 62, 62]
        # ----------------------------------------------------

        x = self.mobilevit(x)

        # ----------------------------------------------------
        # GAP
        # ----------------------------------------------------

        x = self.gap(x).flatten(1)

        # ----------------------------------------------------
        # HANDCRAFTED FEATURES
        # ----------------------------------------------------

        if (
            self.use_handcrafted
            and feat is not None
        ):

            x = torch.cat(
                [x, feat],
                dim=1,
            )

        return self.head(
            x
        ).squeeze(1)


# ============================================================
# SANITY CHECK
# ============================================================

test_model = CNNBase2MobileViTv2(
    in_channels=11,
    n_handcrafted=15,
    dropout=0.35,
    use_se_blocks=True,
    use_handcrafted=True,
).to(DEVICE)

test_model.eval()

x_test = torch.randn(
    2,
    11,
    250,
    250,
    device=DEVICE,
)

hc_test = torch.randn(
    2,
    15,
    device=DEVICE,
)

with torch.no_grad():

    cnn_features = test_model.encoder(
        x_test
    )

    mobilevit_features = test_model.mobilevit(
        cnn_features
    )

    logits = test_model(
        x_test,
        hc_test,
    )


print("=" * 70)
print("CNN_BASE2 → MOBILEVIT-v2")
print("=" * 70)

print(
    "Input:",
    tuple(x_test.shape)
)

print(
    "CNN final feature map:",
    tuple(cnn_features.shape)
)

print(
    "MobileViT output:",
    tuple(mobilevit_features.shape)
)

print(
    "Handcrafted features:",
    tuple(hc_test.shape)
)

print(
    "Final logits:",
    tuple(logits.shape)
)

print(
    "All finite:",
    (
        torch.isfinite(
            cnn_features
        ).all()
        and
        torch.isfinite(
            mobilevit_features
        ).all()
        and
        torch.isfinite(
            logits
        ).all()
    ).item()
)

print("=" * 70)

assert cnn_features.shape == (
    2,
    256,
    62,
    62,
)

assert mobilevit_features.shape == (
    2,
    256,
    62,
    62,
)

assert logits.shape == (
    2,
)

assert torch.isfinite(
    logits
).all()

print(" 256-channel CNN verified")
print(" MobileViT receives final CNN feature map")
print(" MobileViT returns 256-channel feature map")
print(" 15 handcrafted features can be fused")
print(" Final output verified")
print("=" * 70)

del test_model
del x_test
del hc_test
del cnn_features
del mobilevit_features
del logits

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

CNN_BASE2 → MOBILEVIT-v2
Input: (2, 11, 250, 250)
CNN final feature map: (2, 256, 62, 62)
MobileViT output: (2, 256, 62, 62)
Handcrafted features: (2, 15)
Final logits: (2,)
All finite: True
 256-channel CNN verified
 MobileViT receives final CNN feature map
 MobileViT returns 256-channel feature map
 15 handcrafted features can be fused
 Final output verified


In [42]:
# ==============================================================
# CREATE 5-FOLD SPATIAL CROSS-VALIDATION INDICES
# ==============================================================

cv_splitter = StratifiedGroupKFold(
    n_splits=CONFIG["n_folds"],
    shuffle=True,
    random_state=CONFIG["fold_seed"]
)

fold_indices = []

for fold_number, (train_idx, val_idx) in enumerate(
    cv_splitter.split(
        trainval_fp,
        trainval_labels,
        groups=cv_groups
    ),
    start=1
):
    train_groups = set(cv_groups[train_idx])
    val_groups = set(cv_groups[val_idx])

    overlap = train_groups & val_groups

    if len(overlap) != 0:
        raise RuntimeError(
            f"Spatial leakage detected in Fold {fold_number}: {overlap}"
        )

    fold_indices.append(
        (
            train_idx,
            val_idx
        )
    )

print("=" * 70)
print("5-FOLD SPATIAL CROSS-VALIDATION INDICES CREATED")
print("=" * 70)

for fold_number, (train_idx, val_idx) in enumerate(
    fold_indices,
    start=1
):
    print(
        f"Fold {fold_number}: "
        f"Train={len(train_idx)} | "
        f"Validation={len(val_idx)}"
    )

print()
print("Total CV samples:", len(trainval_fp))
print("Number of folds:", len(fold_indices))
print("Spatial leakage: NONE")
print("=" * 70)

5-FOLD SPATIAL CROSS-VALIDATION INDICES CREATED
Fold 1: Train=2475 | Validation=642
Fold 2: Train=2606 | Validation=511
Fold 3: Train=2422 | Validation=695
Fold 4: Train=2472 | Validation=645
Fold 5: Train=2493 | Validation=624

Total CV samples: 3117
Number of folds: 5
Spatial leakage: NONE


In [44]:
print("=" * 70)
print("CNN_BASE2 + MOBILEVIT-v2")
print("5-FOLD SPATIAL CROSS-VALIDATION TRAINING")
print("=" * 70)

oof_probs = np.zeros(
    len(trainval_fp),
    dtype=np.float32
)

oof_labels = trainval_lb_arr.copy()

fold_test_probs = []
fold_val_aucs = []

for fold_i, (tr_idx, va_idx) in enumerate(
    fold_indices,
    start=1
):
    print()
    print("=" * 70)
    print(f"FOLD {fold_i}/{len(fold_indices)}")
    print("=" * 70)

    fold_train_fp = [
        trainval_fp[i]
        for i in tr_idx
    ]

    fold_train_lb = [
        trainval_lb_arr[i]
        for i in tr_idx
    ]

    fold_val_fp = [
        trainval_fp[i]
        for i in va_idx
    ]

    fold_val_lb = [
        trainval_lb_arr[i]
        for i in va_idx
    ]

    fold_train_ds = TerrainPatchDataset(
        fold_train_fp,
        fold_train_lb,
        CHANNEL_MEAN,
        CHANNEL_STD,
        HANDCRAFTED_MEAN,
        HANDCRAFTED_STD,
        augment=True,
        use_handcrafted=CONFIG[
            "use_handcrafted_features"
        ],
        use_channel_dropout=CONFIG[
            "use_channel_dropout"
        ],
        channel_dropout_prob=CONFIG[
            "channel_dropout_prob"
        ],
        channel_dropout_max_channels=CONFIG[
            "channel_dropout_max_channels"
        ],
    )

    fold_val_ds = TerrainPatchDataset(
        fold_val_fp,
        fold_val_lb,
        CHANNEL_MEAN,
        CHANNEL_STD,
        HANDCRAFTED_MEAN,
        HANDCRAFTED_STD,
        augment=False,
        use_handcrafted=CONFIG[
            "use_handcrafted_features"
        ],
    )

    if CONFIG["use_balanced_sampler"]:
        fold_train_lb_arr = np.array(
            fold_train_lb
        )

        class_counts = np.bincount(
            fold_train_lb_arr
        )

        class_weights = (
            1.0 / class_counts
        )

        sample_weights = (
            class_weights[
                fold_train_lb_arr
            ]
        )

        sampler = WeightedRandomSampler(
            sample_weights,
            num_samples=len(
                sample_weights
            ),
            replacement=True,
        )

        fold_train_loader = DataLoader(
            fold_train_ds,
            batch_size=CONFIG[
                "batch_size"
            ],
            sampler=sampler,
            num_workers=CONFIG[
                "num_workers"
            ],
            pin_memory=True,
            drop_last=True,
        )

    else:
        fold_train_loader = DataLoader(
            fold_train_ds,
            batch_size=CONFIG[
                "batch_size"
            ],
            shuffle=True,
            num_workers=CONFIG[
                "num_workers"
            ],
            pin_memory=True,
            drop_last=True,
        )

    fold_val_loader = DataLoader(
        fold_val_ds,
        batch_size=CONFIG[
            "batch_size"
        ],
        shuffle=False,
        num_workers=CONFIG[
            "num_workers"
        ],
        pin_memory=True,
    )

    fold_model = CNNBase2MobileViTv2(
        in_channels=CONFIG["n_channels"],
        n_handcrafted=CONFIG["n_handcrafted"],
        dropout=CONFIG["dropout"],
        use_se_blocks=CONFIG["use_se_blocks"],
        use_handcrafted=CONFIG[
            "use_handcrafted_features"
        ],
    ).to(DEVICE)

    fold_model.apply(
        init_mobilevit_weights
    )

    pos_weight = torch.tensor(
        CONFIG["pos_weight_value"],
        device=DEVICE
    )

    fold_criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    fold_optimizer = torch.optim.AdamW(
        fold_model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"]
    )

    fold_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        fold_optimizer,
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6
    )

    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(
        1,
        CONFIG["epochs"] + 1
    ):
        fold_model.train()

        running_loss = 0.0

        for x, hc, y in fold_train_loader:

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            hc = hc.to(
                DEVICE,
                non_blocking=True
            )

            y = y.float().to(
                DEVICE,
                non_blocking=True
            )

            fold_optimizer.zero_grad(
                set_to_none=True
            )

            with torch.cuda.amp.autocast(
                enabled=(DEVICE.type == "cuda")
            ):
                logits = fold_model(
                    x,
                    hc
                ).view(-1)

                loss = fold_criterion(
                    logits,
                    y
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                fold_model.parameters(),
                1.0
            )

            fold_optimizer.step()

            running_loss += (
                loss.item()
                * x.size(0)
            )

        fold_model.eval()

        val_probs = []
        val_labels = []

        with torch.no_grad():

            for x, hc, y in fold_val_loader:

                x = x.to(
                    DEVICE,
                    non_blocking=True
                )

                hc = hc.to(
                    DEVICE,
                    non_blocking=True
                )

                with torch.cuda.amp.autocast(
                    enabled=(DEVICE.type == "cuda")
                ):
                    logits = fold_model(
                        x,
                        hc
                    ).view(-1)

                probs = torch.sigmoid(
                    logits
                )

                val_probs.append(
                    probs.cpu().numpy()
                )

                val_labels.append(
                    y.numpy()
                )

        val_probs = np.concatenate(
            val_probs
        )

        val_labels = np.concatenate(
            val_labels
        )

        val_auc = roc_auc_score(
            val_labels,
            val_probs
        )

        fold_scheduler.step(
            val_auc
        )

        if val_auc > best_auc:

            best_auc = val_auc
            best_epoch = epoch

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in
                fold_model.state_dict().items()
            }

            patience_counter = 0

        else:
            patience_counter += 1

        if epoch == 1 or epoch % 5 == 0:

            print(
                f"Fold {fold_i} | "
                f"Epoch {epoch:02d} | "
                f"Train Loss "
                f"{running_loss / len(fold_train_ds):.4f} | "
                f"Val AUC "
                f"{val_auc:.4f} | "
                f"Best AUC "
                f"{best_auc:.4f}"
            )

        if (
            patience_counter
            >= CONFIG["early_stopping_patience"]
        ):
            print(
                f"Early stopping at epoch "
                f"{epoch}"
            )
            break

    fold_model.load_state_dict(
        best_state
    )

    fold_model.to(DEVICE)
    fold_model.eval()
    final_val_probs = []

    with torch.no_grad():

        for x, hc, y in fold_val_loader:

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            hc = hc.to(
                DEVICE,
                non_blocking=True
            )

            with torch.cuda.amp.autocast(
                enabled=(DEVICE.type == "cuda")
            ):
                logits = fold_model(
                    x,
                    hc
                ).view(-1)

            final_val_probs.append(
                torch.sigmoid(
                    logits
                ).cpu().numpy()
            )

    final_val_probs = np.concatenate(
        final_val_probs
    )

    final_val_auc = roc_auc_score(
        fold_val_lb,
        final_val_probs
    )

    oof_probs[va_idx] = (
        final_val_probs
    )

    fold_val_aucs.append(
        final_val_auc
    )

    print()
    print(
        f"Fold {fold_i} finished"
    )

    print(
        f"Best AUC: {best_auc:.4f}"
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    del fold_model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


print()
print("=" * 70)
print("5-FOLD TRAINING COMPLETE")
print("=" * 70)

for i, auc in enumerate(
    fold_val_aucs,
    start=1
):
    print(
        f"Fold {i} | AUC {auc:.4f}"
    )

print()
print(
    f"Mean fold AUC: "
    f"{np.mean(fold_val_aucs):.4f}"
)

print(
    f"Fold AUC std: "
    f"{np.std(fold_val_aucs):.4f}"
)

oof_auc = roc_auc_score(
    oof_labels,
    oof_probs
)

print(
    f"OOF ROC-AUC: "
    f"{oof_auc:.4f}"
)

print(
    f"OOF predictions: "
    f"{len(oof_probs)}"
)

print(
    "Held-out test samples used: 0"
)

print("=" * 70)

CNN_BASE2 + MOBILEVIT-v2
5-FOLD SPATIAL CROSS-VALIDATION TRAINING

FOLD 1/5


TypeError: TerrainPatchDataset.__init__() missing 1 required positional argument: 'hc_std'

In [ ]:
cv_splitter = StratifiedGroupKFold(
    n_splits=CONFIG["n_folds"],
    shuffle=True,
    random_state=CONFIG["fold_seed"]
)
fold_splits = []
for fold_number, (
    train_idx,
    val_idx
) in enumerate(

    cv_splitter.split(
        trainval_fp,
        trainval_labels,
        groups=cv_groups
    ),
    start=1
):
    train_groups = set(
        cv_groups[train_idx]
    )
    val_groups = set(
        cv_groups[val_idx]
    )
    overlap = (
        train_groups
        &
        val_groups
    )
    if len(overlap) != 0:

        raise RuntimeError(
            f"Spatial leakage detected "
            f"in Fold {fold_number}: "
            f"{overlap}"
        )


    fold_splits.append(
        (
            train_idx,
            val_idx
        )
    )


    print(
        f"Fold {fold_number}: "
        f"train={len(train_idx)} | "
        f"val={len(val_idx)} | "
        f"val positive="
        f"{trainval_labels[val_idx].mean():.4f} | "
        f"val groups="
        f"{len(val_groups)}"
    )

oof_counter = np.zeros(
    len(trainval_fp),
    dtype=np.int32
)


for _, val_idx in fold_splits:

    oof_counter[val_idx] += 1


print("\n" + "=" * 80)

print(
    "OOF COVERAGE"
)

print(
    "Total samples:",
    len(trainval_fp)
)

print(
    "Validated exactly once:",
    np.sum(
        oof_counter == 1
    )
)

print(
    "Validation assignments:",
    oof_counter.sum()
)

print("=" * 80)
assert np.all(
    oof_counter == 1
)

In [ ]:
# ============================================================
# CELL 18 — FINAL WIDE V5 CNN
# ============================================================

class ConvBNReLU(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size=3,
        padding=1
    ):

        super().__init__()

        self.conv = nn.Conv2d(
            in_ch,
            out_ch,
            kernel_size,
            padding=padding,
            bias=False
        )

        self.bn = nn.BatchNorm2d(
            out_ch
        )

        self.relu = nn.ReLU(
            inplace=True
        )

    def forward(self, x):

        return self.relu(
            self.bn(
                self.conv(x)
            )
        )

class SEBlock(nn.Module):

    def __init__(
        self,
        channels,
        reduction=8
    ):

        super().__init__()

        reduced = max(
            channels // reduction,
            4
        )

        self.pool = nn.AdaptiveAvgPool2d(
            1
        )

        self.fc1 = nn.Linear(
            channels,
            reduced
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.fc2 = nn.Linear(
            reduced,
            channels
        )

        self.sigmoid = nn.Sigmoid()


    def forward(self, x):

        batch_size, channels, _, _ = x.shape

        y = self.pool(
            x
        ).view(
            batch_size,
            channels
        )

        y = self.fc1(y)

        y = self.relu(y)

        y = self.fc2(y)

        y = self.sigmoid(y)

        y = y.view(
            batch_size,
            channels,
            1,
            1
        )

        return x * y

class ConvBlock(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch,
        use_se=True,
        pool=True
    ):

        super().__init__()

        self.conv1 = ConvBNReLU(
            in_ch,
            out_ch
        )

        self.conv2 = ConvBNReLU(
            out_ch,
            out_ch
        )

        self.se = (
            SEBlock(out_ch)
            if use_se
            else nn.Identity()
        )

        self.pool = (
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
            if pool
            else nn.Identity()
        )


    def forward(self, x):

        x = self.conv1(x)

        x = self.conv2(x)

        x = self.se(x)

        x = self.pool(x)

        return x

class CNNEncoder(nn.Module):

    def __init__(
        self,
        in_channels=11,
        use_se_blocks=True
    ):

        super().__init__()

        self.block1 = ConvBlock(
            in_ch=in_channels,
            out_ch=32,
            use_se=use_se_blocks,
            pool=True
        )


        self.block2 = ConvBlock(
            in_ch=32,
            out_ch=64,
            use_se=use_se_blocks,
            pool=True
        )

        self.block3 = ConvBlock(
            in_ch=64,
            out_ch=128,
            use_se=use_se_blocks,
            pool=False
        )


        self.block4 = ConvBlock(
            in_ch=128,
            out_ch=256,
            use_se=use_se_blocks,
            pool=False
        )


    def forward(self, x):

        x = self.block1(x)

        x = self.block2(x)

        x = self.block3(x)

        x = self.block4(x)

        return x


class TerrainCNNFusion(nn.Module):

    def __init__(
        self,
        in_channels=11,
        n_handcrafted=15,
        dropout=0.35,
        use_se_blocks=True,
        use_handcrafted=True
    ):

        super().__init__()


        self.encoder = CNNEncoder(
            in_channels=in_channels,
            use_se_blocks=use_se_blocks
        )


        self.dropout = nn.Dropout2d(
            dropout
        )


        self.gap = nn.AdaptiveAvgPool2d(
            1
        )


        self.use_handcrafted = (
            use_handcrafted
        )

        fusion_dim = (
            256
            +
            (
                n_handcrafted
                if use_handcrafted
                else 0
            )
        )


        self.head = nn.Sequential(

            nn.Linear(
                fusion_dim,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                64,
                1
            )
        )


    def forward(
        self,
        x,
        feat=None
    ):

        x = self.encoder(x)

        x = self.dropout(x)

        x = self.gap(
            x
        ).flatten(1)


        if (
            self.use_handcrafted
            and feat is not None
        ):

            x = torch.cat(
                [
                    x,
                    feat
                ],
                dim=1
            )

        return self.head(
            x
        ).squeeze(1)

def init_weights(module):

    if isinstance(
        module,
        nn.Conv2d
    ):

        nn.init.kaiming_normal_(
            module.weight,
            mode="fan_out",
            nonlinearity="relu"
        )

        if module.bias is not None:

            nn.init.zeros_(
                module.bias
            )


    elif isinstance(
        module,
        nn.Linear
    ):

        nn.init.kaiming_normal_(
            module.weight,
            mode="fan_in",
            nonlinearity="relu"
        )

        if module.bias is not None:

            nn.init.zeros_(
                module.bias
            )


    elif isinstance(
        module,
        nn.BatchNorm2d
    ):

        nn.init.ones_(
            module.weight
        )

        nn.init.zeros_(
            module.bias
        )

In [ ]:
dummy_model = TerrainCNNFusion(
    in_channels=CONFIG["n_channels"],
    n_handcrafted=CONFIG["n_handcrafted"],
    dropout=CONFIG["dropout"],
    use_se_blocks=CONFIG["use_se_blocks"],
    use_handcrafted=CONFIG["use_handcrafted_features"]
).to(DEVICE)


dummy_model.apply(
    init_weights
)


with torch.no_grad():

    dummy_x = torch.randn(
        2,
        CONFIG["n_channels"],
        CONFIG["patch_size"],
        CONFIG["patch_size"],
        device=DEVICE
    )


    dummy_hc = torch.randn(
        2,
        CONFIG["n_handcrafted"],
        device=DEVICE
    )

    dummy_features = (
        dummy_model.encoder(
            dummy_x
        )
    )


    dummy_output = dummy_model(
        dummy_x,
        dummy_hc
    )

print("=" * 80)
print("FINAL WIDE V5 MODEL SANITY CHECK")
print("=" * 80)

print(
    "Input:",
    dummy_x.shape
)

print(
    "CNN feature map:",
    dummy_features.shape
)

print(
    "Handcrafted:",
    dummy_hc.shape
)

print(
    "Output:",
    dummy_output.shape
)

n_params = sum(
    p.numel()
    for p in dummy_model.parameters()
    if p.requires_grad
)


print(
    f"Trainable parameters: "
    f"{n_params:,}"
)


assert dummy_x.shape == (
    2,
    11,
    250,
    250
)


assert dummy_features.shape == (
    2,
    256,
    62,
    62
)


assert dummy_hc.shape == (
    2,
    15
)


assert dummy_output.shape == (
    2,
)


assert torch.all(
    torch.isfinite(
        dummy_features
    )
)


assert torch.all(
    torch.isfinite(
        dummy_output
    )
)

del dummy_model
del dummy_x
del dummy_hc
del dummy_features
del dummy_output


if DEVICE.type == "cuda":

    torch.cuda.empty_cache()


print("=" * 80)
print(
    "FINAL WIDE V5 MODEL SANITY CHECK PASSED."
)
print("=" * 80)

In [ ]:
CONFIG["use_ssl_pretrain"] = True
CONFIG["ssl_epochs"] = 30
CONFIG["ssl_lr"] = 1e-3
CONFIG["ssl_mask_prob"] = 0.25

CONFIG["ssl_checkpoint_path"] = (
    "/kaggle/working/ssl_encoder_256.pt"
)

In [ ]:
print("=" * 80)
print("SELF-SUPERVISED CNN PRETRAINING")
print("=" * 80)

class UnlabeledPatchDataset(Dataset):

    def __init__(
        self,
        filepaths,
        mean,
        std
    ):

        self.filepaths = filepaths

        self.mean = mean.reshape(
            -1, 1, 1
        )

        self.std = std.reshape(
            -1, 1, 1
        )


    def __len__(self):

        return len(
            self.filepaths
        )


    def __getitem__(
        self,
        idx
    ):

        filepath = self.filepaths[
            idx
        ]

        arr = load_patch(
            filepath
        )

        arr = np.ascontiguousarray(
            arr
        )

        arr = (
            arr - self.mean
        ) / self.std

        return torch.from_numpy(
            arr
        ).float()


def gather_unlabeled_pool(
    unlabeled_data_dir,
    fallback_filepaths
):

    if (
        unlabeled_data_dir
        and os.path.isdir(
            unlabeled_data_dir
        )
    ):

        files = sorted(

            glob.glob(
                os.path.join(
                    unlabeled_data_dir,
                    "**",
                    "*.tif"
                ),
                recursive=True
            )

            +

            glob.glob(
                os.path.join(
                    unlabeled_data_dir,
                    "**",
                    "*.tiff"
                ),
                recursive=True
            )
            +

            glob.glob(
                os.path.join(
                    unlabeled_data_dir,
                    "**",
                    "*.npy"
                ),
                recursive=True
            )
        )

        if files:

            print(
                f"Using {len(files)} genuinely "
                f"unlabeled patches for SSL."
            )

            return files


        print(
            "No usable files found in the "
            "unlabeled directory."
        )


    print(
        f"Using train + validation pool "
        f"({len(fallback_filepaths)} patches) "
        f"as the SSL pretraining pool."
    )

    return fallback_filepaths


class SSLDecoder(nn.Module):

    def __init__(
        self,
        out_channels=11
    ):

        super().__init__()


        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(
                256,
                128,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.ConvTranspose2d(
                128,
                64,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                64,
                out_channels,
                kernel_size=3,
                padding=1
            )
        )


    def forward(
        self,
        x
    ):

        x = self.decoder(
            x
        )

        # 248 × 248 → 250 × 250
        x = F.interpolate(
            x,
            size=(250, 250),
            mode="bilinear",
            align_corners=False
        )

        return x


if CONFIG["use_ssl_pretrain"]:

    ssl_pool_fp = gather_unlabeled_pool(
        CONFIG.get(
            "unlabeled_data_dir",
            None
        ),
        trainval_fp
    )


    ssl_dataset = UnlabeledPatchDataset(
        ssl_pool_fp,
        CHANNEL_MEAN,
        CHANNEL_STD
    )


    ssl_loader = DataLoader(
        ssl_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )


    print()
    print(
        f"SSL samples: "
        f"{len(ssl_dataset)}"
    )

    print(
        f"SSL batches per epoch: "
        f"{len(ssl_loader)}"
    )


    ssl_encoder = CNNEncoder(
        in_channels=CONFIG["n_channels"],
        use_se_blocks=CONFIG["use_se_blocks"]
    ).to(
        DEVICE
    )



    ssl_decoder = SSLDecoder(
        out_channels=CONFIG["n_channels"]
    ).to(
        DEVICE
    )

    ssl_encoder.apply(
        init_weights
    )

    ssl_decoder.apply(
        init_weights
    )

    ssl_params = (

        list(
            ssl_encoder.parameters()
        )

        +

        list(
            ssl_decoder.parameters()
        )
    )


    ssl_optimizer = torch.optim.AdamW(
        ssl_params,
        lr=CONFIG["ssl_lr"],
        weight_decay=1e-5
    )

    ssl_criterion = nn.MSELoss()

    ssl_scaler = torch.cuda.amp.GradScaler(
        enabled=(
            DEVICE.type == "cuda"
        )
    )


    print()
    print("=" * 80)

    print(
        f"Starting SSL pretraining "
        f"for {CONFIG['ssl_epochs']} epochs..."
    )

    print(
        f"Mask probability: "
        f"{CONFIG['ssl_mask_prob']}"
    )

    print(
        "Encoder output: "
        "[B, 256, 62, 62]"
    )

    print(
        "Decoder output: "
        "[B, 11, 250, 250]"
    )

    print("=" * 80)


    # ========================================================
    # TRAIN SSL
    # ========================================================

    for epoch in range(
        1,
        CONFIG["ssl_epochs"] + 1
    ):

        ssl_encoder.train()

        ssl_decoder.train()


        running_loss = 0.0

        n_samples = 0


        for clean in tqdm(
            ssl_loader,
            desc=f"SSL Epoch {epoch:02d}",
            leave=False
        ):

            clean = clean.to(
                DEVICE,
                non_blocking=True
            )


            # ------------------------------------------------
            # RANDOM PIXEL MASK
            #
            # Same idea as original v5:
            # mask is shared across channels.
            # ------------------------------------------------

            mask = (
                torch.rand_like(
                    clean[:, :1, :, :]
                )
                >
                CONFIG["ssl_mask_prob"]
            ).float()


            noisy = (
                clean * mask
            )


            ssl_optimizer.zero_grad(
                set_to_none=True
            )


            # ------------------------------------------------
            # FORWARD
            # ------------------------------------------------

            with torch.cuda.amp.autocast(
                enabled=(
                    DEVICE.type == "cuda"
                )
            ):

                feature_map = (
                    ssl_encoder(
                        noisy
                    )
                )


                reconstruction = (
                    ssl_decoder(
                        feature_map
                    )
                )

                loss = ssl_criterion(
                    reconstruction,
                    clean
                )

            # ------------------------------------------------
            # BACKPROP
            # ------------------------------------------------
            ssl_scaler.scale(
                loss
            ).backward()

            ssl_scaler.unscale_(
                ssl_optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                ssl_encoder.parameters(),
                1.0
            )

            torch.nn.utils.clip_grad_norm_(
                ssl_decoder.parameters(),
                1.0
            )

            ssl_scaler.step(
                ssl_optimizer
            )

            ssl_scaler.update()
            # ------------------------------------------------
            # ACCUMULATE
            # ------------------------------------------------
            running_loss += (
                loss.item()
                *
                clean.size(0)
            )
            n_samples += (
                clean.size(0)
            )


        epoch_loss = (
            running_loss
            /
            n_samples
        )
        print(
            f"SSL Epoch {epoch:03d} | "
            f"Reconstruction MSE: "
            f"{epoch_loss:.6f}"
        )
    # ========================================================
    # SAVE ENCODER ONLY
    # ========================================================
    torch.save(
        ssl_encoder.state_dict(),
        CONFIG["ssl_checkpoint_path"]
    )
    print()
    print("=" * 80)
    print(
        "SSL PRETRAINING COMPLETE"
    )

    print(
        "Pretrained encoder saved to:"
    )

    print(
        CONFIG["ssl_checkpoint_path"]
    )

    print("=" * 80)
    # --------------------------------------------------------
    # CLEAN MEMORY
    # --------------------------------------------------------
    del ssl_encoder
    del ssl_decoder
    del ssl_optimizer
    del ssl_scaler

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:

    print(
        "SSL pretraining disabled."
    )

In [ ]:
# ============================================================
# CELL 21 — TRAINING / EVALUATION HELPERS
# ============================================================

def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scaler=None,
    train=False,
    use_handcrafted=True
):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0

    all_probs = []
    all_hard_labels = []

    for batch in tqdm(
        loader,
        desc="Training" if train else "Validation",
        leave=False
    ):

        x = batch[0].to(
            DEVICE,
            non_blocking=True
        )

        feat = batch[1].to(
            DEVICE,
            non_blocking=True
        )

        y_soft = batch[2].to(
            DEVICE,
            non_blocking=True
        )

        y_hard = batch[3].to(
            DEVICE,
            non_blocking=True
        )

        if train:

            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.cuda.amp.autocast(
            enabled=(DEVICE.type == "cuda")
        ):

            logits = model(
                x,
                feat if use_handcrafted else None
            )

            loss = criterion(
                logits,
                y_soft
            )

        if train:

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CONFIG["grad_clip_norm"]
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        total_loss += (
            loss.item()
            * x.size(0)
        )

        probs = torch.sigmoid(
            logits
        )

        all_probs.append(
            probs.detach()
            .cpu()
            .numpy()
        )

        all_hard_labels.append(
            y_hard.detach()
            .cpu()
            .numpy()
        )

    avg_loss = (
        total_loss
        /
        len(loader.dataset)
    )

    probs = np.concatenate(
        all_probs
    )

    hard_labels = np.concatenate(
        all_hard_labels
    )

    preds = (
        probs >= 0.5
    ).astype(
        np.int64
    )

    f1 = f1_score(
        hard_labels,
        preds,
        zero_division=0
    )

    try:

        auc = roc_auc_score(
            hard_labels,
            probs
        )

    except ValueError:

        auc = float("nan")

    return (
        avg_loss,
        f1,
        auc,
        probs,
        hard_labels
    )


# ============================================================
# EPOCH METRIC DISPLAY
# ============================================================

def print_epoch_metrics(
    epoch,
    train_loss,
    train_f1,
    train_auc,
    val_loss,
    val_f1,
    val_auc,
    current_lr,
    marker=""
):

    print(
        f"Epoch {epoch:03d} | "
        f"LR {current_lr:.2e} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train F1 {train_f1:.4f} | "
        f"Train AUC {train_auc:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val F1 {val_f1:.4f} | "
        f"Val AUC {val_auc:.4f}"
        f"{marker}"
    )


print("=" * 80)
print("TRAINING / EVALUATION HELPERS READY")
print("=" * 80)

print("Loss target  : SOFT LABEL")
print("AUC/F1 target: HARD BINARY LABEL")
print("Checkpoint   : VALIDATION ROC-AUC")
print("pos_weight   : DISABLED")

print("=" * 80)

In [ ]:
# ============================================================
# CELL 22A — FIND SPATIAL GROUP INFORMATION
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

print("=" * 80)
print("SEARCHING FOR DATASET METADATA")
print("=" * 80)

metadata_files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

metadata_files = [
    f for f in metadata_files
    if os.path.basename(f).startswith("metadata")
]

print("\nMetadata files found:")

for f in metadata_files:
    print(" ", f)


# ------------------------------------------------------------
# Prefer metadata_all.csv
# ------------------------------------------------------------

metadata_path = None

for f in metadata_files:

    if os.path.basename(f) == "metadata_all.csv":

        metadata_path = f

        break


if metadata_path is None:

    raise FileNotFoundError(
        "metadata_all.csv was not found anywhere under "
        "/kaggle/input."
    )


print("\nUsing:")
print(metadata_path)


# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

metadata = pd.read_csv(
    metadata_path
)


print("\nMetadata shape:")
print(metadata.shape)


print("\nMetadata columns:")
print(
    metadata.columns.tolist()
)


# ------------------------------------------------------------
# CHECK SPATIAL GROUP COLUMN
# ------------------------------------------------------------

possible_group_columns = [
    "spatial_group_id",
    "spatial_group",
    "group_id",
    "group"
]


group_column = None

for col in possible_group_columns:

    if col in metadata.columns:

        group_column = col

        break


if group_column is None:

    raise KeyError(
        "No spatial group column found.\n"
        "Expected one of:\n"
        f"{possible_group_columns}\n\n"
        "Available columns:\n"
        f"{metadata.columns.tolist()}"
    )


print("\nSpatial group column:")
print(group_column)


print("\nNumber of unique spatial groups:")
print(
    metadata[group_column].nunique()
)


print("\nMetadata check PASSED.")
print("=" * 80)

In [ ]:
# ============================================================
# CHECK WHICH METADATA COLUMN MATCHES OUR FILES
# ============================================================

print("=" * 80)
print("CHECKING PATCH NAME MATCH")
print("=" * 80)

print("\nFirst 5 trainval files:")
for fp in trainval_fp[:5]:
    print(" ", os.path.basename(fp))

print("\nFirst 5 metadata patch_name values:")
for x in metadata["patch_name"].astype(str).head(5):
    print(" ", x)

print("\nFirst 5 metadata final_path values:")
for x in metadata["final_path"].astype(str).head(5):
    print(" ", x)

trainval_names = {
    os.path.basename(fp)
    for fp in trainval_fp
}

patch_names = set(
    metadata["patch_name"]
    .astype(str)
)

final_names = set(
    metadata["final_path"]
    .astype(str)
    .apply(os.path.basename)
)

print("\nMATCH RESULTS")
print("-" * 80)

print(
    "Matches using patch_name:",
    len(trainval_names & patch_names),
    "/",
    len(trainval_names)
)

print(
    "Matches using final_path basename:",
    len(trainval_names & final_names),
    "/",
    len(trainval_names)
)

print("=" * 80)

In [ ]:
# ============================================================
# BALANCED SPATIAL STRATIFIED 5-FOLD SEARCH
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

print("=" * 80)
print("SEARCHING FOR BALANCED SPATIAL 5-FOLD SPLIT")
print("=" * 80)

N_FOLDS = 5
TARGET_SIZE = len(trainval_fp) / N_FOLDS
TARGET_POS_RATIO = np.mean(trainval_labels)

print(f"Samples: {len(trainval_fp)}")
print(f"Spatial groups: {len(np.unique(trainval_groups))}")
print(f"Target fold size: {TARGET_SIZE:.1f}")
print(f"Overall positive ratio: {TARGET_POS_RATIO:.4f}")

best_score = float("inf")
best_seed = None
best_folds = None

# ------------------------------------------------------------
# Search multiple seeds
# ------------------------------------------------------------

for seed in range(1000):

    sgkf = StratifiedGroupKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=seed
    )

    candidate_folds = list(
        sgkf.split(
            trainval_fp,
            trainval_labels,
            groups=trainval_groups
        )
    )

    score = 0.0

    fold_sizes = []
    fold_pos_ratios = []

    for train_idx, val_idx in candidate_folds:

        val_size = len(val_idx)

        val_pos_ratio = np.mean(
            trainval_labels[val_idx]
        )

        fold_sizes.append(val_size)
        fold_pos_ratios.append(val_pos_ratio)

        size_error = (
            (val_size - TARGET_SIZE)
            / TARGET_SIZE
        ) ** 2

        ratio_error = (
            val_pos_ratio - TARGET_POS_RATIO
        ) ** 2

        score += (
            2.0 * size_error
            + 5.0 * ratio_error
        )

    if score < best_score:
        best_score = score
        best_seed = seed
        best_folds = candidate_folds


# ------------------------------------------------------------
# Lock best folds
# ------------------------------------------------------------

fold_indices = best_folds

print()
print(f"BEST SEED: {best_seed}")
print(f"BEST SCORE: {best_score:.6f}")

print()
print("=" * 80)
print("FINAL 5-FOLD DISTRIBUTION")
print("=" * 80)

for fold_i, (train_idx, val_idx) in enumerate(
    fold_indices,
    start=1
):

    train_groups = set(
        trainval_groups[train_idx]
    )

    val_groups = set(
        trainval_groups[val_idx]
    )

    overlap = (
        train_groups.intersection(
            val_groups
        )
    )

    val_labels = trainval_labels[val_idx]

    val_size = len(val_idx)

    val_pos = np.sum(
        val_labels == 1
    )

    val_neg = np.sum(
        val_labels == 0
    )

    val_pos_ratio = (
        val_pos / val_size
    )

    print(
        f"\nFold {fold_i}"
    )

    print(
        f"  Train samples     : {len(train_idx)}"
    )

    print(
        f"  Validation samples: {val_size}"
    )

    print(
        f"  Validation site   : {val_pos}"
    )

    print(
        f"  Validation no-site: {val_neg}"
    )

    print(
        f"  Positive ratio    : {val_pos_ratio:.4f}"
    )

    print(
        f"  Train groups      : {len(train_groups)}"
    )

    print(
        f"  Validation groups : {len(val_groups)}"
    )

    print(
        f"  Spatial overlap   : {len(overlap)}"
    )

    if overlap:
        raise RuntimeError(
            f"Spatial leakage detected in fold {fold_i}."
        )


# ------------------------------------------------------------
# COMPLETE OOF COVERAGE CHECK
# ------------------------------------------------------------

val_assignment_count = np.zeros(
    len(trainval_fp),
    dtype=np.int32
)

for _, val_idx in fold_indices:
    val_assignment_count[val_idx] += 1

print()
print("=" * 80)
print("OOF COVERAGE CHECK")
print("=" * 80)

print(
    "Total CV samples:",
    len(trainval_fp)
)

print(
    "Samples validated exactly once:",
    np.sum(val_assignment_count == 1)
)

print(
    "Samples validated zero times:",
    np.sum(val_assignment_count == 0)
)

print(
    "Samples validated more than once:",
    np.sum(val_assignment_count > 1)
)

assert np.all(
    val_assignment_count == 1
)

print()
print("Spatial leakage: NONE")
print("OOF coverage: 100%")
print("Held-out test: UNTOUCHED")

print()
print("=" * 80)
print("FINAL SPATIAL 5-FOLD SPLIT LOCKED")
print("=" * 80)

In [46]:
# ============================================================
# V5 TRAINING HELPER
# ============================================================

from sklearn.metrics import (
    f1_score,
    roc_auc_score
)


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    scaler=None,
    train=False,
    use_handcrafted=True
):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_probs = []
    all_labels = []

    for batch in loader:

        x, hc, soft_labels, hard_labels = batch

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        hc = hc.to(
            DEVICE,
            non_blocking=True
        )

        soft_labels = soft_labels.to(
            DEVICE,
            non_blocking=True
        )

        hard_labels = hard_labels.to(
            DEVICE,
            non_blocking=True
        )

        if train:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.cuda.amp.autocast(
            enabled=(DEVICE.type == "cuda")
        ):

            if use_handcrafted:

                logits = model(
                    x,
                    hc
                )

            else:

                logits = model(
                    x,
                    None
                )

            logits = logits.squeeze(
                -1
            )

            loss = criterion(
                logits,
                soft_labels
            )

        if train:

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        total_loss += (
            loss.item()
            * x.size(0)
        )

        probs = torch.sigmoid(
            logits
        )

        all_probs.extend(
            probs.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_labels.extend(
            hard_labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    avg_loss = (
        total_loss
        / len(loader.dataset)
    )

    all_probs = np.asarray(
        all_probs,
        dtype=np.float32
    )

    all_labels = np.asarray(
        all_labels,
        dtype=np.int64
    )

    preds = (
        all_probs >= 0.5
    ).astype(int)

    f1 = f1_score(
        all_labels,
        preds,
        zero_division=0
    )

    try:

        auc = roc_auc_score(
            all_labels,
            all_probs
        )

    except ValueError:

        auc = float("nan")

    return (
        avg_loss,
        f1,
        auc,
        all_probs,
        all_labels
    )


print("=" * 80)
print("V5 TRAINING HELPER READY")
print("=" * 80)

V5 TRAINING HELPER READY


In [ ]:
# ============================================================
# V5 FINAL SPATIAL 5-FOLD TRAINING
# ============================================================
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler

print("=" * 80)
print("V5 FINAL SPATIAL 5-FOLD TRAINING")
print("=" * 80)

# ============================================================
# BASIC CHECKS
# ============================================================
assert len(fold_indices) == 5, (
    f"Expected 5 folds, found {len(fold_indices)}"
)

assert len(trainval_fp) == len(trainval_labels)
assert len(trainval_fp) == len(cv_monument_counts)

print("Number of locked folds :", len(fold_indices))
print("CV samples             :", len(trainval_fp))
print("Held-out test samples  :", len(test_fp))
print()
# ============================================================
# OOF STORAGE
# ============================================================
oof_probs = np.zeros(
    len(trainval_fp),
    dtype=np.float32
)

oof_labels = np.asarray(
    trainval_labels,
    dtype=np.int64
)

oof_counts = np.zeros(
    len(trainval_fp),
    dtype=np.int32
)

fold_val_aucs = []
fold_test_probs = []

# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================
os.makedirs(
    CONFIG["checkpoint_dir"],
    exist_ok=True
)
# ============================================================
# 5-FOLD TRAINING
# ============================================================
for fold_i, (tr_idx, va_idx) in enumerate(
    fold_indices,
    start=1
):

    print("=" * 80)
    print(f"FOLD {fold_i}/5")
    print("=" * 80)

    # --------------------------------------------------------
    # FILES
    # --------------------------------------------------------
    fold_train_fp = [
        trainval_fp[i]
        for i in tr_idx
    ]

    fold_val_fp = [
        trainval_fp[i]
        for i in va_idx
    ]
    # --------------------------------------------------------
    # HARD LABELS
    # --------------------------------------------------------
    fold_train_lb = np.asarray(
        trainval_labels[tr_idx],
        dtype=np.int64
    )
    fold_val_lb = np.asarray(
        trainval_labels[va_idx],
        dtype=np.int64
    )

    # --------------------------------------------------------
    # MONUMENT COUNTS
    # --------------------------------------------------------
    fold_train_counts = np.asarray(
        cv_monument_counts[tr_idx],
        dtype=np.int64
    )
    fold_val_counts = np.asarray(
        cv_monument_counts[va_idx],
        dtype=np.int64
    )

    print(
        "Train samples     :",
        len(fold_train_fp)
    )

    print(
        "Validation samples:",
        len(fold_val_fp)
    )
    # ========================================================
    # DATASETS
    # ========================================================
    fold_train_ds = TerrainPatchDataset(
        fold_train_fp,
        fold_train_lb,
        fold_train_counts,
        CHANNEL_MEAN,
        CHANNEL_STD,
        HANDCRAFTED_MEAN,
        HANDCRAFTED_STD,
        augment=True,
        use_handcrafted=CONFIG[
            "use_handcrafted_features"
        ],
        use_channel_dropout=CONFIG[
            "use_channel_dropout"
        ],
        channel_dropout_prob=CONFIG[
            "channel_dropout_prob"
        ],
        channel_dropout_max_channels=CONFIG[
            "channel_dropout_max_channels"
        ]
    )

    fold_val_ds = TerrainPatchDataset(
        fold_val_fp,
        fold_val_lb,
        fold_val_counts,
        CHANNEL_MEAN,
        CHANNEL_STD,
        HANDCRAFTED_MEAN,
        HANDCRAFTED_STD,
        augment=False,
        use_handcrafted=CONFIG[
            "use_handcrafted_features"
        ],
        use_channel_dropout=False
    )

    # ========================================================
    # TRAIN LOADER
    # ========================================================

    if CONFIG["use_balanced_sampler"]:

        class_counts = np.bincount(
            fold_train_lb,
            minlength=2
        )

        class_weights = (
            1.0 /
            np.maximum(
                class_counts,
                1
            )
        )

        sample_weights = (
            class_weights[
                fold_train_lb
            ]
        )

        sampler = WeightedRandomSampler(
            torch.tensor(
                sample_weights,
                dtype=torch.double
            ),
            num_samples=len(
                sample_weights
            ),
            replacement=True
        )

        fold_train_loader = DataLoader(
            fold_train_ds,
            batch_size=CONFIG["batch_size"],
            sampler=sampler,
            num_workers=CONFIG.get(
                "num_workers",
                0
            ),
            pin_memory=True
        )

    else:

        fold_train_loader = DataLoader(
            fold_train_ds,
            batch_size=CONFIG["batch_size"],
            shuffle=True,
            num_workers=CONFIG.get(
                "num_workers",
                0
            ),
            pin_memory=True
        )

    # ========================================================
    # VALIDATION LOADER
    # ========================================================

    fold_val_loader = DataLoader(
        fold_val_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG.get(
            "num_workers",
            0
        ),
        pin_memory=True
    )

     # Fresh CNN_BASE2 + MobileViT-v2 model for this fold
    fold_model = CNNBase2MobileViTv2(
        in_channels=CONFIG["n_channels"],
        n_handcrafted=CONFIG["n_handcrafted"],
        dropout=CONFIG["dropout"],
        use_se_blocks=CONFIG["use_se_blocks"],
        use_handcrafted=CONFIG["use_handcrafted_features"],
    ).to(DEVICE)
    
    fold_model.apply(
        init_mobilevit_weights
    )

    print("Fresh model created.")

    # ========================================================
    # LOSS
    # ========================================================

    fold_criterion = nn.BCEWithLogitsLoss()

    # ========================================================
    # OPTIMIZER
    # ========================================================

    fold_optimizer = torch.optim.AdamW(
        fold_model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"]
    )

    # ========================================================
    # SCHEDULER
    # ========================================================

    fold_scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            fold_optimizer,
            mode="min",
            factor=CONFIG[
                "scheduler_factor"
            ],
            patience=CONFIG[
                "scheduler_patience"
            ]
        )
    )

    # ========================================================
    # AMP
    # ========================================================

    fold_scaler = torch.cuda.amp.GradScaler(
        enabled=(
            DEVICE.type == "cuda"
        )
    )
    # ========================================================
    # CHECKPOINT
    # ========================================================

    fold_checkpoint_path = os.path.join(
        CONFIG["checkpoint_dir"],
        f"fold_{fold_i}.pt"
    )

    # ========================================================
    # BEST MODEL TRACKING
    # ========================================================

    best_val_auc = -np.inf
    epochs_without_improvement = 0

    # ========================================================
    # TRAIN
    # ========================================================
    for epoch in range(
        1,
        CONFIG["epochs"] + 1
    ):

        # ----------------------------------------------------
        # TRAINING
        # ----------------------------------------------------
        (
            train_loss,
            train_f1,
            train_auc,
            _,
            _
        ) = run_epoch(
            fold_model,
            fold_train_loader,
            fold_criterion,
            optimizer=fold_optimizer,
            scaler=fold_scaler,
            train=True,
            use_handcrafted=CONFIG[
                "use_handcrafted_features"
            ]
        )

        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------
        (
            val_loss,
            val_f1,
            val_auc,
            val_probs,
            val_labels
        ) = run_epoch(
            fold_model,
            fold_val_loader,
            fold_criterion,
            optimizer=None,
            scaler=None,
            train=False,
            use_handcrafted=CONFIG[
                "use_handcrafted_features"
            ]
        )

        # ----------------------------------------------------
        # SCHEDULER
        # ----------------------------------------------------
        fold_scheduler.step(
            val_loss
        )

        current_lr = (
            fold_optimizer
            .param_groups[0]["lr"]
        )

        # ----------------------------------------------------
        # BEST CHECKPOINT = VALIDATION AUC
        # ----------------------------------------------------
        marker = ""
        if (
            np.isfinite(val_auc)
            and
            val_auc > best_val_auc
        ):

            best_val_auc = val_auc

            epochs_without_improvement = 0

            torch.save(
                fold_model.state_dict(),
                fold_checkpoint_path
            )

            marker = " <- BEST AUC"

        else:
            epochs_without_improvement += 1

        # ----------------------------------------------------
        # DISPLAY
        # ----------------------------------------------------

        print(
            f"Fold {fold_i} | "
            f"Epoch {epoch:03d} | "
            f"LR {current_lr:.2e} | "
            f"Train AUC {train_auc:.4f} | "
            f"Val AUC {val_auc:.4f} | "
            f"Val F1 {val_f1:.4f}"
            f"{marker}"
        )

        # ----------------------------------------------------
        # EARLY STOPPING
        # ----------------------------------------------------

        if (
            epochs_without_improvement
            >= CONFIG[
                "early_stopping_patience"
            ]
        ):

            print(
                f"Fold {fold_i}: "
                f"early stopping after "
                f"{CONFIG['early_stopping_patience']} "
                f"epochs without AUC improvement."
            )

            break

    # ========================================================
    # CHECKPOINT MUST EXIST
    # ========================================================
    if not os.path.exists(
        fold_checkpoint_path
    ):

        raise RuntimeError(
            f"Fold {fold_i} checkpoint was not saved."
        )

    print()
    print(
        f"Fold {fold_i} BEST VAL AUC: "
        f"{best_val_auc:.4f}"
    )

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================
    fold_model.load_state_dict(
        torch.load(
            fold_checkpoint_path,
            map_location=DEVICE
        )
    )
    # ========================================================
    # FINAL VALIDATION PREDICTION
    # ========================================================
    (
        final_val_loss,
        final_val_f1,
        final_val_auc,
        final_val_probs,
        final_val_labels
    ) = run_epoch(
        fold_model,
        fold_val_loader,
        fold_criterion,
        optimizer=None,
        scaler=None,
        train=False,
        use_handcrafted=CONFIG[
            "use_handcrafted_features"
        ]
    )

    # --------------------------------------------------------
    # STORE OOF
    # --------------------------------------------------------

    oof_probs[va_idx] = final_val_probs
    oof_labels[va_idx] = final_val_labels
    oof_counts[va_idx] += 1

    fold_val_aucs.append(
        final_val_auc
    )

    print(
        f"Fold {fold_i} final Val AUC: "
        f"{final_val_auc:.4f}"
    )

    print(
        f"Fold {fold_i} OOF predictions stored: "
        f"{len(final_val_probs)}"
    )
    # ========================================================
    # HELD-OUT TEST PREDICTION
    # ========================================================

    test_zero_counts = np.zeros(
        len(test_fp),
        dtype=np.int64
    )

    test_dataset = TerrainPatchDataset(
        test_fp,
        np.asarray(
            test_labels,
            dtype=np.int64
        ),
        test_zero_counts,
        CHANNEL_MEAN,
        CHANNEL_STD,
        HANDCRAFTED_MEAN,
        HANDCRAFTED_STD,
        augment=False,
        use_handcrafted=CONFIG[
            "use_handcrafted_features"
        ],
        use_channel_dropout=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG.get(
            "num_workers",
            0
        ),
        pin_memory=True
    )
    # --------------------------------------------------------
    # TEST PREDICTIONS
    # --------------------------------------------------------

    test_probs_all = []
    fold_model.eval()
    with torch.no_grad():

        for batch in test_loader:
            x = batch[0].to(
                DEVICE,
                non_blocking=True
            )

            hc = batch[1].to(
                DEVICE,
                non_blocking=True
            )

            if CONFIG[
                "use_handcrafted_features"
            ]:

                logits = fold_model(
                    x,
                    hc
                )

            else:

                logits = fold_model(
                    x,
                    None
                )

            probs = torch.sigmoid(
                logits
            )

            test_probs_all.extend(
                probs.cpu()
                .numpy()
                .tolist()
            )

    test_probs_all = np.asarray(
        test_probs_all,
        dtype=np.float32
    )

    fold_test_probs.append(
        test_probs_all
    )

    print(
        f"Fold {fold_i} test predictions: "
        f"{len(test_probs_all)}"
    )

    # ========================================================
    # CLEAN GPU MEMORY
    # ========================================================

    del fold_model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print()
    print(
        f"FOLD {fold_i} COMPLETE."
    )
    print()


# ========================================================================
# FINAL OOF CHECK
# ========================================================================

print("=" * 80)
print("5-FOLD TRAINING COMPLETE")
print("=" * 80)

print(
    "Validation AUC per fold:",
    [
        round(float(x), 4)
        for x in fold_val_aucs
    ]
)

print(
    f"Mean validation AUC: "
    f"{np.nanmean(fold_val_aucs):.4f}"
)

print(
    f"Validation AUC std: "
    f"{np.nanstd(fold_val_aucs):.4f}"
)

print()
print(
    "OOF samples:",
    len(oof_probs)
)

print(
    "Samples with exactly one OOF prediction:",
    np.sum(oof_counts == 1)
)
print(
    "Samples with zero OOF predictions:",
    np.sum(oof_counts == 0)
)

print(
    "Samples with >1 OOF predictions:",
    np.sum(oof_counts > 1)
)

assert np.all(
    oof_counts == 1
), "OOF coverage is not exactly 100%."

assert len(fold_test_probs) == 5

assert all(
    len(x) == len(test_fp)
    for x in fold_test_probs
)

print()
print("OOF coverage: 100%")
print("Spatial leakage: NONE")
print("Held-out test remained untouched during training: YES")
print("5 fold test prediction sets: 5")
print()
print("=" * 80)
print("V5 5-FOLD TRAINING + OOF + TEST PREDICTIONS COMPLETE")
print("=" * 80)

V5 FINAL SPATIAL 5-FOLD TRAINING
Number of locked folds : 5
CV samples             : 3117
Held-out test samples  : 500

FOLD 1/5
Train samples     : 2475
Validation samples: 642
Fresh model created.


/tmp/ipykernel_58/3641672602.py:281: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  fold_scaler = torch.cuda.amp.GradScaler(
/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Fold 1 | Epoch 001 | LR 3.00e-04 | Train AUC 0.5068 | Val AUC 0.4559 | Val F1 0.4293 <- BEST AUC


/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Fold 1 | Epoch 002 | LR 3.00e-04 | Train AUC 0.5131 | Val AUC 0.5554 | Val F1 0.0000 <- BEST AUC


...

/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Fold 5 | Epoch 015 | LR 7.50e-05 | Train AUC 0.5944 | Val AUC 0.5736 | Val F1 0.0000


/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Fold 5 | Epoch 016 | LR 7.50e-05 | Train AUC 0.5895 | Val AUC 0.5515 | Val F1 0.0187


/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


In [ ]:
# ============================================================
# FINAL OOF THRESHOLD SELECTION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

print("=" * 80)
print("FINAL OOF EVALUATION + THRESHOLD SELECTION")
print("=" * 80)

# ------------------------------------------------------------
# OOF ROC-AUC
# ------------------------------------------------------------

oof_auc = roc_auc_score(
    oof_labels,
    oof_probs
)

print(
    f"OOF ROC-AUC: {oof_auc:.4f}"
)

# ------------------------------------------------------------
# SEARCH FOR ONE GLOBAL THRESHOLD
# ------------------------------------------------------------

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

for threshold in thresholds:

    oof_preds = (
        oof_probs >= threshold
    ).astype(int)

    acc = accuracy_score(
        oof_labels,
        oof_preds
    )

    precision = precision_score(
        oof_labels,
        oof_preds,
        zero_division=0
    )

    recall = recall_score(
        oof_labels,
        oof_preds,
        zero_division=0
    )

    f1 = f1_score(
        oof_labels,
        oof_preds,
        zero_division=0
    )

    threshold_results.append(
        (
            threshold,
            acc,
            precision,
            recall,
            f1
        )
    )

# ------------------------------------------------------------
# SELECT THRESHOLD BY BEST OOF F1
# ------------------------------------------------------------

best_threshold, best_oof_acc, best_oof_precision, \
best_oof_recall, best_oof_f1 = max(
    threshold_results,
    key=lambda x: x[4]
)

print()
print("=" * 80)
print("BEST OOF OPERATING POINT")
print("=" * 80)

print(
    f"Best threshold : {best_threshold:.2f}"
)

print(
    f"OOF Accuracy   : {best_oof_acc:.4f}"
)

print(
    f"OOF Precision  : {best_oof_precision:.4f}"
)

print(
    f"OOF Recall     : {best_oof_recall:.4f}"
)

print(
    f"OOF F1         : {best_oof_f1:.4f}"
)

print(
    f"OOF ROC-AUC    : {oof_auc:.4f}"
)

# ------------------------------------------------------------
# OOF CONFUSION MATRIX
# ------------------------------------------------------------

oof_final_preds = (
    oof_probs >= best_threshold
).astype(int)

print()
print("OOF CONFUSION MATRIX")
print("(rows=true, cols=predicted)")
print(
    confusion_matrix(
        oof_labels,
        oof_final_preds
    )
)

# ------------------------------------------------------------
# STORE THRESHOLD
# ------------------------------------------------------------

FINAL_THRESHOLD = float(
    best_threshold
)

print()
print("=" * 80)
print(
    f"FINAL THRESHOLD LOCKED: "
    f"{FINAL_THRESHOLD:.2f}"
)
print(
    "Threshold selected from OOF predictions ONLY."
)
print(
    "Held-out test set was NOT used."
)
print("=" * 80)

In [ ]:
# ============================================================
# FINAL HELD-OUT TEST EVALUATION
# 5-FOLD ENSEMBLE + LOCKED OOF THRESHOLD
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("=" * 80)
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# CHECK THAT ALL 5 FOLD TEST PREDICTIONS EXIST
# ------------------------------------------------------------

assert len(fold_test_probs) == 5, (
    f"Expected 5 fold test prediction sets, "
    f"but found {len(fold_test_probs)}"
)

print(
    f"Number of fold test prediction sets: "
    f"{len(fold_test_probs)}"
)

# ------------------------------------------------------------
# CHECK EACH FOLD HAS 500 TEST PREDICTIONS
# ------------------------------------------------------------

for i, probs in enumerate(fold_test_probs, start=1):

    print(
        f"Fold {i} test predictions: "
        f"{len(probs)}"
    )

    assert len(probs) == len(test_labels), (
        f"Fold {i} prediction count does not match "
        f"test label count."
    )

# ------------------------------------------------------------
# CONVERT TO NUMPY ARRAYS
# ------------------------------------------------------------

fold_test_arrays = [
    np.asarray(p).reshape(-1)
    for p in fold_test_probs
]

# ------------------------------------------------------------
# 5-FOLD ENSEMBLE
# ------------------------------------------------------------

ensemble_test_probs = np.mean(
    np.stack(fold_test_arrays, axis=0),
    axis=0
)

print()
print(
    "Ensemble test predictions:",
    len(ensemble_test_probs)
)

# ------------------------------------------------------------
# FINAL LOCKED THRESHOLD
# ------------------------------------------------------------

final_threshold = FINAL_THRESHOLD

ensemble_test_preds = (
    ensemble_test_probs >= final_threshold
).astype(int)

# ------------------------------------------------------------
# FINAL TEST METRICS
# ------------------------------------------------------------

test_auc = roc_auc_score(
    test_labels,
    ensemble_test_probs
)

test_accuracy = accuracy_score(
    test_labels,
    ensemble_test_preds
)

test_precision = precision_score(
    test_labels,
    ensemble_test_preds,
    zero_division=0
)

test_recall = recall_score(
    test_labels,
    ensemble_test_preds,
    zero_division=0
)

test_f1 = f1_score(
    test_labels,
    ensemble_test_preds,
    zero_division=0
)

# ------------------------------------------------------------
# PRINT FINAL RESULTS
# ------------------------------------------------------------

print()
print("=" * 80)
print("FINAL HELD-OUT TEST RESULTS")
print("=" * 80)

print(
    f"Threshold : {final_threshold:.2f}"
)

print(
    f"Accuracy  : {test_accuracy:.4f} "
    f"({test_accuracy * 100:.2f}%)"
)

print(
    f"Precision : {test_precision:.4f} "
    f"({test_precision * 100:.2f}%)"
)

print(
    f"Recall    : {test_recall:.4f} "
    f"({test_recall * 100:.2f}%)"
)

print(
    f"F1 Score  : {test_f1:.4f} "
    f"({test_f1 * 100:.2f}%)"
)

print(
    f"ROC-AUC   : {test_auc:.4f} "
    f"({test_auc * 100:.2f}%)"
)

# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    test_labels,
    ensemble_test_preds
)

print()
print("CONFUSION MATRIX")
print("(rows = true, columns = predicted)")
print(cm)

print()
print("=" * 80)
print("FINAL TEST EVALUATION COMPLETE")
print("=" * 80)

print(
    "Test set size:",
    len(test_labels)
)

print(
    "Test set was never used for threshold selection."
)

print(
    "Threshold came from OOF predictions only."
)

print(
    "Final prediction = mean of 5 fold predictions."
)

print("=" * 80)

In [ ]:
# ============================================================
# V5 FINAL DIAGNOSTIC SUMMARY — CORRECTED
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 80)
print("V5 MODEL DIAGNOSTIC SUMMARY")
print("=" * 80)

# ============================================================
# 1. PER-FOLD VALIDATION AUC
# ============================================================

print()
print("PER-FOLD VALIDATION AUC")
print()

for i, auc in enumerate(fold_val_aucs, start=1):
    print(f"Fold {i}: {auc:.4f}")

print()
print(f"Mean validation AUC : {np.mean(fold_val_aucs):.4f}")
print(f"Std validation AUC  : {np.std(fold_val_aucs):.4f}")
print(f"Best fold AUC       : {np.max(fold_val_aucs):.4f}")
print(f"Worst fold AUC      : {np.min(fold_val_aucs):.4f}")

# ============================================================
# 2. OOF PERFORMANCE
# ============================================================

print()
print("OOF PERFORMANCE")
print()

print(f"OOF ROC-AUC      : {oof_auc:.4f}")
print(f"Locked threshold : {best_threshold:.2f}")

oof_preds = (
    np.asarray(oof_probs) >= best_threshold
).astype(int)

oof_accuracy = accuracy_score(
    oof_labels,
    oof_preds
)

oof_precision = precision_score(
    oof_labels,
    oof_preds,
    zero_division=0
)

oof_recall = recall_score(
    oof_labels,
    oof_preds,
    zero_division=0
)

oof_f1 = f1_score(
    oof_labels,
    oof_preds,
    zero_division=0
)

print(f"OOF Accuracy     : {oof_accuracy:.4f}")
print(f"OOF Precision    : {oof_precision:.4f}")
print(f"OOF Recall       : {oof_recall:.4f}")
print(f"OOF F1           : {oof_f1:.4f}")

# ============================================================
# 3. OOF PROBABILITY DISTRIBUTION
# ============================================================

print()
print("OOF PROBABILITY DISTRIBUTION")
print()

oof_probs_np = np.asarray(oof_probs).reshape(-1)
oof_labels_np = np.asarray(oof_labels).reshape(-1)

print(f"Minimum probability : {np.min(oof_probs_np):.4f}")
print(f"25th percentile     : {np.percentile(oof_probs_np, 25):.4f}")
print(f"Median              : {np.median(oof_probs_np):.4f}")
print(f"75th percentile     : {np.percentile(oof_probs_np, 75):.4f}")
print(f"Maximum probability : {np.max(oof_probs_np):.4f}")

print()
print(
    f"Predicted positive rate : "
    f"{np.mean(oof_preds):.4f}"
)

print(
    f"Actual positive rate    : "
    f"{np.mean(oof_labels_np):.4f}"
)

# ============================================================
# 4. BEST SINGLE FOLD VS ENSEMBLE
# ============================================================

print()
print("BEST SINGLE FOLD VS 5-FOLD ENSEMBLE")
print()

best_fold_idx = int(
    np.argmax(fold_val_aucs)
)

single_fold_probs = np.asarray(
    fold_test_probs[best_fold_idx]
).reshape(-1)

single_fold_preds = (
    single_fold_probs >= best_threshold
).astype(int)

single_fold_auc = roc_auc_score(
    test_labels,
    single_fold_probs
)

single_fold_f1 = f1_score(
    test_labels,
    single_fold_preds,
    zero_division=0
)

print(
    f"Best fold               : Fold {best_fold_idx + 1}"
)

print(
    f"Best fold test ROC-AUC  : {single_fold_auc:.4f}"
)

print(
    f"Best fold test F1       : {single_fold_f1:.4f}"
)

print(
    f"5-fold ensemble ROC-AUC : {test_auc:.4f}"
)

print(
    f"5-fold ensemble F1      : {test_f1:.4f}"
)

print(
    f"AUC ensemble gain       : "
    f"{test_auc - single_fold_auc:+.4f}"
)

print(
    f"F1 ensemble gain        : "
    f"{test_f1 - single_fold_f1:+.4f}"
)

# ============================================================
# 5. FINAL TEST CONFUSION MATRIX
# ============================================================

print()
print("HELD-OUT TEST CONFUSION MATRIX")
print()

print(
    confusion_matrix(
        test_labels,
        ensemble_test_preds
    )
)

# ============================================================
# 6. FOLD AUC VISUALIZATION
# ============================================================

plt.figure(figsize=(7, 4))

plt.bar(
    range(1, len(fold_val_aucs) + 1),
    fold_val_aucs
)

plt.axhline(
    np.mean(fold_val_aucs),
    linestyle="--",
    label=f"Mean = {np.mean(fold_val_aucs):.4f}"
)

plt.xlabel("Fold")
plt.ylabel("Validation ROC-AUC")
plt.title("Spatial 5-Fold Validation AUC")
plt.xticks(
    range(1, len(fold_val_aucs) + 1)
)
plt.legend()
plt.show()

# ============================================================
# 7. OOF PROBABILITY VISUALIZATION
# ============================================================

plt.figure(figsize=(7, 4))

plt.hist(
    oof_probs_np[oof_labels_np == 0],
    bins=30,
    alpha=0.6,
    label="No-site"
)

plt.hist(
    oof_probs_np[oof_labels_np == 1],
    bins=30,
    alpha=0.6,
    label="Site"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Threshold = {best_threshold:.2f}"
)

plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("OOF Probability Distribution")
plt.legend()
plt.show()

# ============================================================
# COMPLETE
# ============================================================

print()
print("=" * 80)
print("DIAGNOSTIC SUMMARY COMPLETE")
print("=" * 80)

In [52]:
# ============================================================
# FOLD 5 — FINAL VALIDATION FROM SAVED CHECKPOINT
# ============================================================

fold_model.eval()

# Loss function is required by run_epoch even during evaluation
eval_criterion = nn.BCEWithLogitsLoss()

(
    fold5_val_loss,
    fold5_val_f1,
    fold5_val_auc,
    fold5_val_probs,
    fold5_val_labels,
) = run_epoch(
    fold_model,
    fold_val_loader,
    eval_criterion,          # <-- FIX
    optimizer=None,
    scaler=None,
    train=False,
    use_handcrafted=CONFIG["use_handcrafted_features"],
)

print()
print("=" * 80)
print("FOLD 5 FINAL VALIDATION")
print("=" * 80)

print(f"Validation Loss : {fold5_val_loss:.4f}")
print(f"Validation F1   : {fold5_val_f1:.4f}")
print(f"Validation AUC  : {fold5_val_auc:.4f}")
print(f"Predictions     : {len(fold5_val_probs)}")
print("=" * 80)

/tmp/ipykernel_58/1576861305.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



FOLD 5 FINAL VALIDATION
Validation Loss : 0.6178
Validation F1   : 0.0724
Validation AUC  : 0.5992
Predictions     : 624


In [53]:
# ============================================================
# FOLD 5 → OOF STORAGE + INTEGRITY CHECK
# NO TRAINING
# ============================================================

import numpy as np

print("=" * 80)
print("FOLD 5 → OOF STORAGE")
print("=" * 80)

# Fold 5 validation indices
fold5_train_idx, fold5_val_idx = fold_indices[4]

print("Fold 5 validation samples expected :", len(fold5_val_idx))
print("Fold 5 predictions available       :", len(fold5_val_probs))
print("Fold 5 labels available            :", len(fold5_val_labels))

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(fold5_val_idx) == len(fold5_val_probs), (
    f"Index/prediction mismatch: "
    f"{len(fold5_val_idx)} vs {len(fold5_val_probs)}"
)

assert len(fold5_val_probs) == len(fold5_val_labels), (
    f"Prediction/label mismatch: "
    f"{len(fold5_val_probs)} vs {len(fold5_val_labels)}"
)

# ------------------------------------------------------------
# Store Fold 5 predictions in the correct OOF locations
# ------------------------------------------------------------

oof_probs[fold5_val_idx] = np.asarray(
    fold5_val_probs,
    dtype=np.float32
)

oof_labels[fold5_val_idx] = np.asarray(
    fold5_val_labels,
    dtype=np.int64
)

oof_counts[fold5_val_idx] += 1

print()
print("Fold 5 OOF predictions stored:", len(fold5_val_probs))

# ------------------------------------------------------------
# Check OOF coverage
# ------------------------------------------------------------

print()
print("=" * 80)
print("OOF COVERAGE CHECK")
print("=" * 80)

print("Total CV samples       :", len(oof_probs))
print("Exactly 1 prediction  :", np.sum(oof_counts == 1))
print("Zero predictions      :", np.sum(oof_counts == 0))
print(">1 predictions        :", np.sum(oof_counts > 1))

print()

if np.all(oof_counts == 1):
    print("✅ OOF COVERAGE = 100%")
    print("✅ Every CV sample has exactly one prediction.")
else:
    print("⚠️ OOF COVERAGE IS NOT COMPLETE YET.")

print("=" * 80)

FOLD 5 → OOF STORAGE
Fold 5 validation samples expected : 624
Fold 5 predictions available       : 624
Fold 5 labels available            : 624

Fold 5 OOF predictions stored: 624

OOF COVERAGE CHECK
Total CV samples       : 3117
Exactly 1 prediction  : 2493
Zero predictions      : 0
>1 predictions        : 624

⚠️ OOF COVERAGE IS NOT COMPLETE YET.


In [54]:
# ============================================================
# FIX DUPLICATE FOLD 5 OOF COUNT
# ============================================================

print("=" * 80)
print("FIXING DUPLICATE FOLD 5 OOF STORAGE")
print("=" * 80)

fold5_train_idx, fold5_val_idx = fold_indices[4]

print("Fold 5 samples:", len(fold5_val_idx))
print("Counts BEFORE fix:")
print("  Exactly 1:", np.sum(oof_counts == 1))
print("  >1      :", np.sum(oof_counts > 1))
print("  0       :", np.sum(oof_counts == 0))

# Remove the extra Fold 5 count
oof_counts[fold5_val_idx] -= 1

print()
print("Counts AFTER fix:")
print("  Exactly 1:", np.sum(oof_counts == 1))
print("  >1      :", np.sum(oof_counts > 1))
print("  0       :", np.sum(oof_counts == 0))

print()

assert np.all(oof_counts == 1), (
    "OOF coverage is still not exactly 100%."
)

print("✅ OOF COVERAGE = 100%")
print("✅ Every one of the 3117 CV samples has exactly ONE prediction.")
print("✅ No Fold 5 retraining required.")
print("=" * 80)

FIXING DUPLICATE FOLD 5 OOF STORAGE
Fold 5 samples: 624
Counts BEFORE fix:
  Exactly 1: 2493
  >1      : 624
  0       : 0

Counts AFTER fix:
  Exactly 1: 3117
  >1      : 0
  0       : 0

✅ OOF COVERAGE = 100%
✅ Every one of the 3117 CV samples has exactly ONE prediction.
✅ No Fold 5 retraining required.


In [55]:
# ============================================================
# OVERALL 5-FOLD OOF RESULTS
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("=" * 80)
print("OVERALL 5-FOLD OOF RESULTS")
print("=" * 80)

# Safety check
assert np.all(oof_counts == 1), "OOF coverage is not 100%."

# ------------------------------------------------------------
# OOF labels and probabilities
# ------------------------------------------------------------

y_true = oof_labels.astype(np.int64)
y_prob = oof_probs.astype(np.float32)

# Convert probability → binary prediction
y_pred = (y_prob >= 0.5).astype(np.int64)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

oof_accuracy = accuracy_score(y_true, y_pred)

oof_precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

oof_recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

oof_f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

oof_auc = roc_auc_score(
    y_true,
    y_prob
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred
)

tn, fp, fn, tp = cm.ravel()

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print()
print("Number of OOF samples :", len(y_true))
print()

print(f"Accuracy              : {oof_accuracy:.4f}")
print(f"Precision             : {oof_precision:.4f}")
print(f"Recall                : {oof_recall:.4f}")
print(f"F1 Score              : {oof_f1:.4f}")
print(f"ROC-AUC               : {oof_auc:.4f}")

print()
print("=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(cm)

print()
print(f"True Negatives (TN)   : {tn}")
print(f"False Positives (FP)  : {fp}")
print(f"False Negatives (FN)  : {fn}")
print(f"True Positives (TP)   : {tp}")

print()
print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Negative", "Positive"],
        zero_division=0
    )
)

print("=" * 80)

OVERALL 5-FOLD OOF RESULTS

Number of OOF samples : 3117

Accuracy              : 0.6978
Precision             : 0.5667
Recall                : 0.0180
F1 Score              : 0.0348
ROC-AUC               : 0.5662

CONFUSION MATRIX
[[2158   13]
 [ 929   17]]

True Negatives (TN)   : 2158
False Positives (FP)  : 13
False Negatives (FN)  : 929
True Positives (TP)   : 17

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Negative       0.70      0.99      0.82      2171
    Positive       0.57      0.02      0.03       946

    accuracy                           0.70      3117
   macro avg       0.63      0.51      0.43      3117
weighted avg       0.66      0.70      0.58      3117



In [59]:
# ============================================================
# RESTORE ORIGINAL TTA FUNCTION
# This is the TTA implementation used by the original notebook.
# ============================================================

def tta_predict(
    model,
    filepaths,
    mean,
    std,
    hc_mean,
    hc_std,
    device,
    use_handcrafted=True,
):
    model.eval()

    mean_r = mean.reshape(-1, 1, 1)
    std_r = std.reshape(-1, 1, 1)

    all_probs = []

    transforms = []

    # Original notebook: 8 TTA transformations
    for flip_h in [False, True]:
        for flip_v in [False, True]:
            for k in [0, 1]:
                transforms.append(
                    (flip_h, flip_v, k)
                )

    with torch.no_grad():

        for fp in tqdm(
            filepaths,
            desc="TTA inference",
            leave=False,
        ):

            raw_arr = load_patch(fp)

            # ------------------------------------------------
            # Handcrafted features
            # ------------------------------------------------
            if use_handcrafted:

                hc = compute_handcrafted_features(
                    raw_arr
                )

                hc = (
                    hc - hc_mean
                ) / hc_std

                hc_t = (
                    torch.from_numpy(hc)
                    .float()
                    .unsqueeze(0)
                    .to(device)
                )

            else:
                hc_t = None

            probs_this_patch = []

            # ------------------------------------------------
            # 8 TTA transformations
            # ------------------------------------------------
            for flip_h, flip_v, k in transforms:

                a = raw_arr.copy()

                if flip_h:
                    a = a[:, :, ::-1]

                if flip_v:
                    a = a[:, ::-1, :]

                if k:
                    a = np.rot90(
                        a,
                        k=k,
                        axes=(1, 2),
                    )

                # Make array contiguous after flips/rotation
                a = np.ascontiguousarray(a)

                # Normalise channels
                a = (
                    a - mean_r
                ) / std_r

                x = (
                    torch.from_numpy(a)
                    .float()
                    .unsqueeze(0)
                    .to(device)
                )

                with torch.cuda.amp.autocast(
                    enabled=(device.type == "cuda")
                ):

                    logit = model(
                        x,
                        hc_t,
                    )

                probs_this_patch.append(
                    torch.sigmoid(
                        logit
                    ).item()
                )

            # Average all 8 TTA predictions
            all_probs.append(
                np.mean(
                    probs_this_patch
                )
            )

    return np.array(
        all_probs
    )

print(" tta_predict() restored successfully.")

 tta_predict() restored successfully.


In [63]:
# ============================================================
# FOLD 5 — COMPLETE HELD-OUT TEST PREDICTIONS
# NO TRAINING
# ============================================================

print("=" * 80)
print("FOLD 5 RECOVERY — HELD-OUT TEST PREDICTIONS")
print("=" * 80)

# ------------------------------------------------------------
# Load Fold 5 best checkpoint
# ------------------------------------------------------------

fold5_model = CNNBase2MobileViTv2(
    in_channels=CONFIG["n_channels"],
    n_handcrafted=CONFIG["n_handcrafted"],
    dropout=CONFIG["dropout"],
    use_se_blocks=CONFIG["use_se_blocks"],
    use_handcrafted=CONFIG["use_handcrafted_features"],
).to(DEVICE)

fold5_checkpoint_path = os.path.join(
    CONFIG["checkpoint_dir"],
    "fold_5.pt"
)

fold5_model.load_state_dict(
    torch.load(
        fold5_checkpoint_path,
        map_location=DEVICE,
    )
)

fold5_model.eval()

print(" Fold 5 best checkpoint loaded")

# ------------------------------------------------------------
# Generate Fold 5 test predictions
# ------------------------------------------------------------

fold5_test_probs = tta_predict(
    fold5_model,
    test_fp,
    CHANNEL_MEAN,
    CHANNEL_STD,
    HANDCRAFTED_MEAN,
    HANDCRAFTED_STD,
    DEVICE,
    use_handcrafted=CONFIG[
        "use_handcrafted_features"
    ],
)

fold5_test_probs = np.asarray(
    fold5_test_probs,
    dtype=np.float32,
)

print()
print("Fold 5 test predictions generated:", len(fold5_test_probs))

# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

assert len(fold5_test_probs) == len(test_fp)

assert np.isfinite(
    fold5_test_probs
).all()

print(" Exactly 500 Fold 5 test predictions generated.")

# ------------------------------------------------------------
# Put Fold 5 into fold_test_probs
# ------------------------------------------------------------

# We already have Folds 1–4.
# Remove an accidental existing Fold 5 if present.

if len(fold_test_probs) >= 5:
    fold_test_probs = fold_test_probs[:4]

assert len(fold_test_probs) == 4, (
    f"Expected Folds 1–4, "
    f"but found {len(fold_test_probs)}."
)

fold_test_probs.append(
    fold5_test_probs
)

# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print()
print("=" * 80)
print("FOLD TEST PREDICTION COVERAGE")
print("=" * 80)

print(
    "Number of folds:",
    len(fold_test_probs)
)

for i, probs in enumerate(
    fold_test_probs,
    start=1,
):
    print(
        f"Fold {i}: "
        f"{len(probs)} test predictions"
    )

assert len(fold_test_probs) == 5

assert all(
    len(probs) == len(test_fp)
    for probs in fold_test_probs
)

print()
print(" Fold 1: 500 predictions")
print(" Fold 2: 500 predictions")
print(" Fold 3: 500 predictions")
print(" Fold 4: 500 predictions")
print(" Fold 5: 500 predictions")
print()
print(" Fold 5 successfully recovered.")
print(" NO Fold 5 retraining.")
print(" NO Fold 1–4 retraining.")
print("=" * 80)

FOLD 5 RECOVERY — HELD-OUT TEST PREDICTIONS
 Fold 5 best checkpoint loaded


TTA inference:   0%|          | 0/500 [00:00<?, ?it/s]

/tmp/ipykernel_58/2578291814.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



Fold 5 test predictions generated: 500
 Exactly 500 Fold 5 test predictions generated.

FOLD TEST PREDICTION COVERAGE
Number of folds: 5
Fold 1: 500 test predictions
Fold 2: 500 test predictions
Fold 3: 500 test predictions
Fold 4: 500 test predictions
Fold 5: 500 test predictions

 Fold 1: 500 predictions
 Fold 2: 500 predictions
 Fold 3: 500 predictions
 Fold 4: 500 predictions
 Fold 5: 500 predictions

 Fold 5 successfully recovered.
 NO Fold 5 retraining.
 NO Fold 1–4 retraining.


In [65]:
# ================================================================
# FINAL 5-FOLD ENSEMBLE EVALUATION
# ================================================================
# IMPORTANT:
# - NO TRAINING
# - NO RETRAINING
# - Uses saved predictions from Folds 1-5
# - Fold 5 is the recovered checkpoint prediction
# ================================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("=" * 80)
print("FINAL 5-FOLD ENSEMBLE EVALUATION")
print("=" * 80)


# ================================================================
# 1. BASIC SAFETY CHECKS
# ================================================================

print("\n[1] CHECKING STORED PREDICTIONS")
print("-" * 80)

assert "fold_test_probs" in globals(), \
    "fold_test_probs is not available."

assert "test_labels" in globals(), \
    "test_labels is not available."

assert len(fold_test_probs) == 5, \
    f"Expected 5 folds, found {len(fold_test_probs)}."

for i, probs in enumerate(fold_test_probs, start=1):

    probs = np.asarray(probs)

    print(
        f"Fold {i}: "
        f"{len(probs)} predictions | "
        f"min={probs.min():.4f} | "
        f"max={probs.max():.4f}"
    )

    assert len(probs) == len(test_labels), (
        f"Fold {i} has {len(probs)} predictions, "
        f"but test set has {len(test_labels)} labels."
    )

print("\n All 5 folds have predictions for all test samples.")
print(f" Held-out test samples: {len(test_labels)}")


# ================================================================
# 2. CONVERT TO NUMPY
# ================================================================

test_labels_np = np.asarray(test_labels).astype(int)

fold_test_probs_np = [
    np.asarray(p, dtype=np.float64)
    for p in fold_test_probs
]


# ================================================================
# 3. CREATE 5-FOLD ENSEMBLE
# ================================================================
# Each test sample has 5 probability predictions:
#
# Fold 1 ─┐
# Fold 2 ─┤
# Fold 3 ─┼──> MEAN ──> FINAL TEST PROBABILITY
# Fold 4 ─┤
# Fold 5 ─┘
#
# No test labels are used here.
# ================================================================

ensemble_test_probs = np.mean(
    np.stack(fold_test_probs_np, axis=0),
    axis=0
)

print("\n[2] 5-FOLD ENSEMBLE CREATED")
print("-" * 80)

print(f"Number of test samples : {len(ensemble_test_probs)}")
print(
    f"Probability range      : "
    f"{ensemble_test_probs.min():.4f} → "
    f"{ensemble_test_probs.max():.4f}"
)
print(
    f"Mean probability       : "
    f"{ensemble_test_probs.mean():.4f}"
)


# ================================================================
# 4. CHECK OOF PREDICTIONS
# ================================================================
# OOF = Out-Of-Fold predictions.
#
# Every CV sample must have exactly ONE prediction.
# These predictions are used to determine the threshold.
#
# IMPORTANT:
# We do NOT determine the threshold using the held-out test set.
# ================================================================

print("\n[3] CHECKING OOF PREDICTIONS")
print("-" * 80)

assert "oof_probs" in globals(), \
    "oof_probs is not available."

assert "oof_labels" in globals(), \
    "oof_labels is not available."

oof_probs_np = np.asarray(oof_probs, dtype=np.float64)
oof_labels_np = np.asarray(oof_labels).astype(int)

print(f"OOF samples : {len(oof_probs_np)}")
print(f"OOF labels  : {len(oof_labels_np)}")

assert len(oof_probs_np) == len(oof_labels_np), \
    "OOF probabilities and labels have different lengths."

print("✓ OOF predictions and labels match.")


# ================================================================
# 5. OOF ROC-AUC
# ================================================================

oof_auc = roc_auc_score(
    oof_labels_np,
    oof_probs_np
)

print(f"\nOOF ROC-AUC : {oof_auc:.4f}")


# ================================================================
# 6. FIND THRESHOLD USING OOF DATA ONLY
# ================================================================
#
# We search thresholds from 0.05 to 0.95.
#
# The held-out test set is NOT used to choose the threshold.
# This is important because the test set must remain unseen.
#
# We select the threshold giving the best OOF F1.
# ================================================================

thresholds = np.arange(0.05, 0.951, 0.01)

best_threshold = 0.50
best_oof_f1 = -1.0

threshold_results = []

for threshold in thresholds:

    oof_pred = (
        oof_probs_np >= threshold
    ).astype(int)

    f1 = f1_score(
        oof_labels_np,
        oof_pred,
        zero_division=0
    )

    precision = precision_score(
        oof_labels_np,
        oof_pred,
        zero_division=0
    )

    recall = recall_score(
        oof_labels_np,
        oof_pred,
        zero_division=0
    )

    threshold_results.append(
        (threshold, precision, recall, f1)
    )

    if f1 > best_oof_f1:

        best_oof_f1 = f1
        best_threshold = threshold


print("\n[4] OOF THRESHOLD SELECTION")
print("-" * 80)

print(f"Best OOF threshold : {best_threshold:.2f}")
print(f"Best OOF F1        : {best_oof_f1:.4f}")


# ================================================================
# 7. SHOW TOP OOF THRESHOLDS
# ================================================================

threshold_results_sorted = sorted(
    threshold_results,
    key=lambda x: x[3],
    reverse=True
)

print("\nTop 10 OOF thresholds:")
print(
    f"{'Threshold':>10} "
    f"{'Precision':>12} "
    f"{'Recall':>10} "
    f"{'F1':>10}"
)

for threshold, precision, recall, f1 in threshold_results_sorted[:10]:

    print(
        f"{threshold:10.2f} "
        f"{precision:12.4f} "
        f"{recall:10.4f} "
        f"{f1:10.4f}"
    )


# ================================================================
# 8. FINAL HELD-OUT TEST PREDICTIONS
# ================================================================

final_test_pred = (
    ensemble_test_probs >= best_threshold
).astype(int)


# ================================================================
# 9. FINAL TEST METRICS
# ================================================================

test_accuracy = accuracy_score(
    test_labels_np,
    final_test_pred
)

test_precision = precision_score(
    test_labels_np,
    final_test_pred,
    zero_division=0
)

test_recall = recall_score(
    test_labels_np,
    final_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    test_labels_np,
    final_test_pred,
    zero_division=0
)

test_auc = roc_auc_score(
    test_labels_np,
    ensemble_test_probs
)


# ================================================================
# 10. CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    test_labels_np,
    final_test_pred
)

tn, fp, fn, tp = cm.ravel()


# ================================================================
# 11. FINAL RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("FINAL HELD-OUT TEST RESULTS — 5-FOLD ENSEMBLE")
print("=" * 80)

print(f"\nNumber of test samples : {len(test_labels_np)}")
print(f"Decision threshold     : {best_threshold:.2f}")

print("\nMETRICS")
print("-" * 80)

print(f"Accuracy               : {test_accuracy:.4f}")
print(f"Precision              : {test_precision:.4f}")
print(f"Recall                 : {test_recall:.4f}")
print(f"F1 Score               : {test_f1:.4f}")
print(f"ROC-AUC                : {test_auc:.4f}")


print("\nCONFUSION MATRIX")
print("-" * 80)

print(cm)

print(f"\nTrue Negatives (TN)    : {tn}")
print(f"False Positives (FP)   : {fp}")
print(f"False Negatives (FN)   : {fn}")
print(f"True Positives (TP)    : {tp}")


# ================================================================
# 12. CLASSIFICATION REPORT
# ================================================================

print("\nCLASSIFICATION REPORT")
print("-" * 80)

print(
    classification_report(
        test_labels_np,
        final_test_pred,
        target_names=["Negative", "Positive"],
        digits=4,
        zero_division=0
    )
)


# ================================================================
# 13. COMPARE AGAINST STANDARD 0.50 THRESHOLD
# ================================================================
#
# This is only a diagnostic comparison.
# Our primary result above uses the threshold selected from OOF.
# ================================================================

pred_05 = (
    ensemble_test_probs >= 0.50
).astype(int)

acc_05 = accuracy_score(
    test_labels_np,
    pred_05
)

prec_05 = precision_score(
    test_labels_np,
    pred_05,
    zero_division=0
)

rec_05 = recall_score(
    test_labels_np,
    pred_05,
    zero_division=0
)

f1_05 = f1_score(
    test_labels_np,
    pred_05,
    zero_division=0
)


print("\n")
print("=" * 80)
print("DIAGNOSTIC COMPARISON — THRESHOLD 0.50")
print("=" * 80)

print(f"Accuracy   : {acc_05:.4f}")
print(f"Precision  : {prec_05:.4f}")
print(f"Recall     : {rec_05:.4f}")
print(f"F1 Score   : {f1_05:.4f}")
print(f"ROC-AUC    : {test_auc:.4f}")


# ================================================================
# 14. FINAL SANITY CHECK
# ================================================================

print("\n")
print("=" * 80)
print("FINAL SANITY CHECK")
print("=" * 80)

assert len(ensemble_test_probs) == 500, \
    "Final ensemble should contain 500 test predictions."

assert len(final_test_pred) == 500, \
    "Final prediction array should contain 500 predictions."

print(" 5 fold predictions used")
print(" 500 held-out test samples")
print(" Fold 5 recovered from saved checkpoint")
print(" No retraining performed")
print(" Threshold selected from OOF data")
print(" Final metrics calculated on held-out test set")
print("=" * 80)

FINAL 5-FOLD ENSEMBLE EVALUATION

[1] CHECKING STORED PREDICTIONS
--------------------------------------------------------------------------------
Fold 1: 500 predictions | min=0.2285 | max=0.8074
Fold 2: 500 predictions | min=0.2374 | max=0.6580
Fold 3: 500 predictions | min=0.0476 | max=0.4251
Fold 4: 500 predictions | min=0.1546 | max=0.6347
Fold 5: 500 predictions | min=0.2172 | max=0.9735

 All 5 folds have predictions for all test samples.
 Held-out test samples: 500

[2] 5-FOLD ENSEMBLE CREATED
--------------------------------------------------------------------------------
Number of test samples : 500
Probability range      : 0.1792 → 0.6714
Mean probability       : 0.3480

[3] CHECKING OOF PREDICTIONS
--------------------------------------------------------------------------------
OOF samples : 3117
OOF labels  : 3117
✓ OOF predictions and labels match.

OOF ROC-AUC : 0.5662

[4] OOF THRESHOLD SELECTION
--------------------------------------------------------------------------